In [1]:
import pandas as pd

In [2]:

# Load all datasets

stores = pd.read_csv("data/dark_stores_messy.csv")
partners = pd.read_csv("data/delivery_partners_messy.csv")
order_items = pd.read_csv("data/order_items_messy.csv")
orders = pd.read_csv("data/orders_clean.csv")

products = pd.read_csv("data/products_messy.csv")
promos = pd.read_csv("data/promo_codes_messy.csv")
store_hours = pd.read_csv("data/store_hours_messy.csv")
trips = pd.read_csv("data/trip_logs_messy.csv")

print("All datasets loaded successfully!")

All datasets loaded successfully!


In [3]:
datasets = {
    "Stores": stores,
    "Partners": partners,
    "Order Items": order_items,
    "Orders": orders,
    "Products": products,
    "Promos": promos,
    "Store Hours": store_hours,
    "Trips": trips
}

for name, df in datasets.items():
    print(f"{name}: {df.shape}")

Stores: (12, 9)
Partners: (124, 7)
Order Items: (177902, 6)
Orders: (52133, 14)
Products: (54, 7)
Promos: (6, 7)
Store Hours: (67, 4)
Trips: (52445, 7)


In [4]:
# Phase 1B - Data Quality Audit

for name, df in datasets.items():
    print("\n" + "="*70)
    print(f"{name.upper()}")
    print("="*70)

    print("Rows, Columns:", df.shape)
    print("Duplicate rows:", df.duplicated().sum())

    print("\nMissing values:")
    missing = df.isnull().sum()
    print(missing[missing > 0])

    print("\nData types:")
    print(df.dtypes)


STORES
Rows, Columns: (12, 9)
Duplicate rows: 2

Missing values:
staff_count    2
dtype: int64

Data types:
store_id                  object
store_name                object
pincode                    int64
opened_date               object
sqft                       int64
monthly_rent              object
staff_count              float64
cold_storage_capacity      int64
city                      object
dtype: object

PARTNERS
Rows, Columns: (124, 7)
Duplicate rows: 0

Missing values:
joined_date    12
dtype: int64

Data types:
partner_id         object
partner_name       object
joined_date        object
vehicle_type       object
home_store_id      object
employment_type    object
rating             object
dtype: object

ORDER ITEMS
Rows, Columns: (177902, 6)
Duplicate rows: 864

Missing values:
unit_cost    2134
dtype: int64

Data types:
order_item_id     object
order_id          object
sku               object
quantity           int64
unit_price       float64
unit_cost        float64


In [5]:
# Check uniqueness of important IDs

print("Stores")
print("Rows:", len(stores))
print("Unique store IDs:", stores["store_id"].nunique())

print("\nOrders")
print("Rows:", len(orders))
print("Unique order IDs:", orders["order_id"].nunique())

print("\nTrips")
print("Rows:", len(trips))
print("Unique trip IDs:", trips["trip_id"].nunique())
print("Unique order IDs:", trips["order_id"].nunique())

print("\nPartners")
print("Rows:", len(partners))
print("Unique partner IDs:", partners["partner_id"].nunique())

print("\nProducts")
print("Rows:", len(products))
print("Unique SKUs:", products["sku"].nunique())

Stores
Rows: 12
Unique store IDs: 10

Orders
Rows: 52133
Unique order IDs: 52133

Trips
Rows: 52445
Unique trip IDs: 52133
Unique order IDs: 52133

Partners
Rows: 124
Unique partner IDs: 124

Products
Rows: 54
Unique SKUs: 54


In [6]:
stores

,store_id,store_name,pincode,opened_date,sqft,monthly_rent,staff_count,cold_storage_capacity,city
0,S01,Sector 18 Hub,201301,2025-06-15,2200,"Rs. 178,000",14.0,900,Noida
1,S02,Sector 62 Express,201309,01/07/2025,1900,"INR 162,000",12.0,800,NOIDA
2,S03,Sector 137 Riverside,201305,10-Aug-2025,2100,"196,000",13.0,850,noida
3,S04,Greater Noida Alpha,201310,2025-09-05,2000,"Rs. 152,000",12.0,820,Nodia
4,S05,Sector 76 Central,201304,20/09/2025,1800,"INR 181,000",11.0,700,Noida
5,S06,Noida Extension North,201306,11-Oct-2025,2300,"158,000",15.0,950,Noida
6,S07,Sector 44 Metro,201303,2025-06-28,2400,"Rs. 228,000",16.0,380,NOIDA
7,S08,Sector 104 Gardens,201304,02/11/2025,1700,"INR 146,000",NaN,700,noida
8,S09,Sector 150 Sports City,201310,15-Jun-2026,2150,"172,000",13.0,830,Nodia
9,S10,Sector 51 Junction,201301,2026-02-08,1850,"Rs. 165,000",12.0,760,Noida


In [7]:
stores.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   store_id               12 non-null     object 
 1   store_name             12 non-null     object 
 2   pincode                12 non-null     int64  
 3   opened_date            12 non-null     object 
 4   sqft                   12 non-null     int64  
 5   monthly_rent           12 non-null     object 
 6   staff_count            10 non-null     float64
 7   cold_storage_capacity  12 non-null     int64  
 8   city                   12 non-null     object 
dtypes: float64(1), int64(3), object(5)
memory usage: 996.0+ bytes


In [8]:
stores["store_id"].value_counts()

store_id
S08    2
S04    2
S02    1
S01    1
S05    1
S03    1
S06    1
S07    1
S09    1
S10    1
Name: count, dtype: int64

## cleaning log

In [9]:
cleaning_log = pd.DataFrame(columns=[
    "Dataset",
    "Column",
    "Issue",
    "Rows_Affected",
    "Action",
    "Reason",
    "Impact_on_Analysis"
])

cleaning_log

,Dataset,Column,Issue,Rows_Affected,Action,Reason,Impact_on_Analysis


In [10]:
cleaning_log = pd.DataFrame([
    {
        "Dataset": "dark_stores_messy",
        "Column": "All columns",
        "Issue": "Duplicate store records",
        "Rows_Affected": 2,
        "Action": "Remove exact duplicate records",
        "Reason": "S04 and S08 appear twice; store master should contain one record per store",
        "Impact_on_Analysis": "Prevents duplicated store information from affecting store-level analysis"
    },
    {
        "Dataset": "dark_stores_messy",
        "Column": "staff_count",
        "Issue": "Missing values",
        "Rows_Affected": 2,
        "Action": "Investigate before filling",
        "Reason": "Staff count is a store attribute; arbitrary imputation could distort operational analysis",
        "Impact_on_Analysis": "May affect orders-per-staff or productivity analysis"
    },
    {
        "Dataset": "dark_stores_messy",
        "Column": "monthly_rent",
        "Issue": "Inconsistent text format",
        "Rows_Affected": 12,
        "Action": "Convert to numeric",
        "Reason": "Values contain formats such as Rs. and INR",
        "Impact_on_Analysis": "Required for store P&L calculation"
    },
    {
        "Dataset": "dark_stores_messy",
        "Column": "opened_date",
        "Issue": "Multiple date formats",
        "Rows_Affected": 12,
        "Action": "Standardize to datetime",
        "Reason": "Dates appear in formats such as YYYY-MM-DD, DD/MM/YYYY and DD-Mon-YYYY",
        "Impact_on_Analysis": "Ensures consistent date calculations"
    },
    {
        "Dataset": "delivery_partners_messy",
        "Column": "joined_date",
        "Issue": "Missing values",
        "Rows_Affected": 12,
        "Action": "Investigate before filling",
        "Reason": "Missing joining dates cannot safely be inferred",
        "Impact_on_Analysis": "May affect partner-tenure analysis"
    },
    {
        "Dataset": "order_items_messy",
        "Column": "All columns",
        "Issue": "Duplicate records",
        "Rows_Affected": 864,
        "Action": "Investigate and remove confirmed duplicates",
        "Reason": "Duplicate order-item records can inflate quantities and costs",
        "Impact_on_Analysis": "Directly affects product sales and COGS analysis"
    },
    {
        "Dataset": "order_items_messy",
        "Column": "unit_cost",
        "Issue": "Missing values",
        "Rows_Affected": 2134,
        "Action": "Reconcile with product master",
        "Reason": "Product master contains unit cost and can potentially provide the correct value",
        "Impact_on_Analysis": "Affects product-level COGS calculation"
    },
    {
        "Dataset": "orders_clean",
        "Column": "delivered_ts",
        "Issue": "Missing values",
        "Rows_Affected": 1067,
        "Action": "Investigate against order status",
        "Reason": "Missing delivery timestamps may be valid for cancelled orders",
        "Impact_on_Analysis": "Important for delivery-time and SLA analysis"
    },
    {
        "Dataset": "orders_clean",
        "Column": "promo_code",
        "Issue": "Missing values",
        "Rows_Affected": 40067,
        "Action": "Interpret as no promo only after validation",
        "Reason": "Missing promo code may represent orders without a promotion",
        "Impact_on_Analysis": "Important for promotion impact analysis"
    },
    {
        "Dataset": "trip_logs_messy",
        "Column": "All columns",
        "Issue": "Duplicate records",
        "Rows_Affected": 312,
        "Action": "Remove confirmed duplicate trip records",
        "Reason": "Trip table should have one trip record per trip",
        "Impact_on_Analysis": "Prevents duplicate trips from distorting delivery metrics"
    },
    {
        "Dataset": "trip_logs_messy",
        "Column": "delivered_ts",
        "Issue": "Missing values",
        "Rows_Affected": 1072,
        "Action": "Investigate against trip_status",
        "Reason": "Missing delivery time may be valid for unsuccessful trips",
        "Impact_on_Analysis": "Affects delivery-time and SLA analysis"
    }
])

cleaning_log

,Dataset,Column,Issue,Rows_Affected,Action,Reason,Impact_on_Analysis
0,dark_stores_messy,All columns,Duplicate store records,2,Remove exact duplicate records,S04 and S08 appear twice; store master should ...,Prevents duplicated store information from aff...
1,dark_stores_messy,staff_count,Missing values,2,Investigate before filling,Staff count is a store attribute; arbitrary im...,May affect orders-per-staff or productivity an...
2,dark_stores_messy,monthly_rent,Inconsistent text format,12,Convert to numeric,Values contain formats such as Rs. and INR,Required for store P&L calculation
3,dark_stores_messy,opened_date,Multiple date formats,12,Standardize to datetime,"Dates appear in formats such as YYYY-MM-DD, DD...",Ensures consistent date calculations
4,delivery_partners_messy,joined_date,Missing values,12,Investigate before filling,Missing joining dates cannot safely be inferred,May affect partner-tenure analysis
5,order_items_messy,All columns,Duplicate records,864,Investigate and remove confirmed duplicates,Duplicate order-item records can inflate quant...,Directly affects product sales and COGS analysis
6,order_items_messy,unit_cost,Missing values,2134,Reconcile with product master,Product master contains unit cost and can pote...,Affects product-level COGS calculation
7,orders_clean,delivered_ts,Missing values,1067,Investigate against order status,Missing delivery timestamps may be valid for c...,Important for delivery-time and SLA analysis
8,orders_clean,promo_code,Missing values,40067,Interpret as no promo only after validation,Missing promo code may represent orders withou...,Important for promotion impact analysis
9,trip_logs_messy,All columns,Duplicate records,312,Remove confirmed duplicate trip records,Trip table should have one trip record per trip,Prevents duplicate trips from distorting deliv...


In [11]:
# Orders with missing delivery timestamp
orders[orders["delivered_ts"].isna()]["order_status"].value_counts()

order_status
cancelled    1067
Name: count, dtype: int64

In [12]:
# Orders with missing promo code
orders[orders["promo_code"].isna()]["order_status"].value_counts()

order_status
delivered    37414
returned      1853
cancelled      800
Name: count, dtype: int64

In [13]:
# Missing product costs
missing_cost = order_items[order_items["unit_cost"].isna()]

print("Missing unit_cost rows:", len(missing_cost))

print("\nMissing-cost SKUs:")
print(missing_cost["sku"].value_counts())

Missing unit_cost rows: 2134

Missing-cost SKUs:
sku
SKU0047    63
SKU0037    61
SKU0034    57
SKU0031    57
SKU0052    55
SKU0051    54
SKU0054    54
SKU0042    54
SKU0049    54
SKU0050    53
SKU0036    52
SKU0040    52
SKU0053    49
SKU0044    48
SKU0032    47
SKU0039    46
SKU0030    45
SKU0046    45
SKU0015    45
SKU0048    44
SKU0038    44
SKU0033    44
SKU0043    43
SKU0002    41
SKU0008    41
SKU0045    40
SKU0035    39
SKU0041    38
SKU0007    38
SKU0029    35
SKU0018    34
SKU0023    34
SKU0022    33
SKU0019    32
SKU0024    32
SKU0005    32
SKU0017    31
SKU0026    31
SKU0014    30
SKU0006    30
SKU0013    30
SKU0016    29
SKU0012    29
SKU0003    29
SKU0027    28
SKU0020    28
SKU0001    28
SKU0004    28
SKU0010    27
SKU0011    27
SKU0009    27
SKU0021    25
SKU0025    22
SKU0028    20
Name: count, dtype: int64


In [14]:
missing_cost.merge(
    products[["sku", "unit_cost"]],
    on="sku",
    how="left",
    suffixes=("_order_item", "_product")
)[["sku", "unit_cost_order_item", "unit_cost_product"]].head(20)

,sku,unit_cost_order_item,unit_cost_product
0,SKU0026,NaN,136.79
1,SKU0039,NaN,43.54
2,SKU0037,NaN,60.88
3,SKU0047,NaN,37.04
4,SKU0053,NaN,108.35
5,SKU0041,NaN,52.50
6,SKU0051,NaN,42.47
7,SKU0032,NaN,102.68
8,SKU0044,NaN,36.27
9,SKU0048,NaN,76.93


In [15]:
# Trip statuses with missing delivery timestamp
trips[trips["delivered_ts"].isna()]["trip_status"].value_counts()

trip_status
cancelled    1072
Name: count, dtype: int64

In [16]:
# check negative distance

print("Negative distances:", (trips["distance_km"] < 0).sum())

trips[trips["distance_km"] < 0][
    ["trip_id", "order_id", "partner_id", "distance_km", "trip_status"]
].head(20)

Negative distances: 577


,trip_id,order_id,partner_id,distance_km,trip_status
6,T000007,O000007,P0009,-0.80,delivered
75,T000076,O000076,P0006,-2.08,delivered
78,T000079,O000079,P0001,-2.27,delivered
155,T000156,O000156,P0022,-1.98,delivered
337,T000338,O000338,P0032,-3.78,delivered
540,T000541,O000541,P0052,-4.51,delivered
677,T000678,O000678,P0068,-1.31,returned
780,T000781,O000781,P0075,-4.76,delivered
808,T000809,O000809,P0074,-0.82,delivered
813,T000814,O000814,P0073,-1.07,delivered


In [17]:
missing_promo = orders[orders["promo_code"].isna()]

print("Missing promo orders:", len(missing_promo))

print("\nDiscount amount summary:")
print(missing_promo["discount_amount"].describe())

print("\nOrders with missing promo but non-zero discount:")
print(
    (missing_promo["discount_amount"] > 0).sum()
)

Missing promo orders: 40067

Discount amount summary:
count    40067.0
mean         0.0
std          0.0
min          0.0
25%          0.0
50%          0.0
75%          0.0
max          0.0
Name: discount_amount, dtype: float64

Orders with missing promo but non-zero discount:
0


In [18]:
print(
    missing_promo.groupby("order_status")["discount_amount"]
    .agg(["count", "sum", "mean", "max"])
)

              count  sum  mean  max
order_status                       
cancelled       800  0.0   0.0  0.0
delivered     37414  0.0   0.0  0.0
returned       1853  0.0   0.0  0.0


In [19]:
# investigate the negative distances
negative_distance = trips[trips["distance_km"] < 0].copy()

print("Negative-distance trips:", len(negative_distance))

print("\nBy trip status:")
print(negative_distance["trip_status"].value_counts())

print("\nDistance statistics:")
print(negative_distance["distance_km"].describe())

Negative-distance trips: 577

By trip status:
trip_status
delivered    525
returned      31
cancelled     21
Name: count, dtype: int64

Distance statistics:
count    577.000000
mean      -3.022236
std        1.469443
min       -5.400000
25%       -4.410000
50%       -3.040000
75%       -1.670000
max       -0.600000
Name: distance_km, dtype: float64


In [20]:
negative_distance["abs_distance"] = negative_distance["distance_km"].abs()

print(negative_distance["abs_distance"].describe())

count    577.000000
mean       3.022236
std        1.469443
min        0.600000
25%        1.670000
50%        3.040000
75%        4.410000
max        5.400000
Name: abs_distance, dtype: float64


In [21]:
print(
    negative_distance.groupby("trip_status")["abs_distance"]
    .agg(["count", "mean", "median", "min", "max"])
)

             count      mean  median   min   max
trip_status                                     
cancelled       21  2.654286    2.33  0.94  5.37
delivered      525  3.010838    3.01  0.60  5.40
returned        31  3.464516    3.81  0.89  5.36


In [22]:
# Check the missing staff count
stores[stores["staff_count"].isna()]

,store_id,store_name,pincode,opened_date,sqft,monthly_rent,staff_count,cold_storage_capacity,city
7,S08,Sector 104 Gardens,201304,02/11/2025,1700,"INR 146,000",NaN,700,noida
11,S08,Sector 104 Gardens,201304,02/11/2025,1700,"INR 146,000",NaN,700,noida


In [23]:
partners.groupby("home_store_id").size().sort_index()

home_store_id
S01    11
S02    11
S03    13
S04    11
S05    13
S06    13
S07    13
S08    13
S09    15
S10    11
dtype: int64

In [24]:
# Create clean copies of all datasets

stores_clean = stores.copy()
partners_clean = partners.copy()
order_items_clean = order_items.copy()
orders_final = orders.copy()
products_clean = products.copy()
promos_clean = promos.copy()
store_hours_clean = store_hours.copy()
trips_clean = trips.copy()

print("Clean dataset copies created successfully!")

Clean dataset copies created successfully!


In [25]:
trips_clean["distance_km"] = trips_clean["distance_km"].abs()

In [26]:
# ==============================
# CLEAN THE DATASETS
# ==============================

# 1. STORES
stores_clean = stores_clean.drop_duplicates()

stores_clean["monthly_rent"] = (
    stores_clean["monthly_rent"]
    .astype(str)
    .str.replace("Rs.", "", regex=False)
    .str.replace("INR", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
    .astype(float)
)

stores_clean["opened_date"] = pd.to_datetime(
    stores_clean["opened_date"],
    dayfirst=True,
    errors="coerce"
)

stores_clean["city"] = stores_clean["city"].str.strip().str.title()


# 2. PARTNERS
partners_clean["joined_date"] = pd.to_datetime(
    partners_clean["joined_date"],
    dayfirst=True,
    errors="coerce"
)


# 3. ORDER ITEMS
order_items_clean = order_items_clean.drop_duplicates()

order_items_clean = order_items_clean.merge(
    products_clean[["sku", "unit_cost"]].rename(
        columns={"unit_cost": "product_unit_cost"}
    ),
    on="sku",
    how="left"
)

order_items_clean["unit_cost"] = (
    order_items_clean["unit_cost"]
    .fillna(order_items_clean["product_unit_cost"])
)

order_items_clean = order_items_clean.drop(
    columns=["product_unit_cost"]
)


# 4. ORDERS
orders_final["order_ts"] = pd.to_datetime(
    orders_final["order_ts"],
    errors="coerce"
)

orders_final["delivered_ts"] = pd.to_datetime(
    orders_final["delivered_ts"],
    errors="coerce"
)

orders_final["promo_code"] = (
    orders_final["promo_code"]
    .fillna("NO_PROMO")
)


# 5. PROMOS
promos_clean["valid_from"] = pd.to_datetime(
    promos_clean["valid_from"],
    dayfirst=True,
    errors="coerce"
)

promos_clean["valid_to"] = pd.to_datetime(
    promos_clean["valid_to"],
    dayfirst=True,
    errors="coerce"
)


# 6. TRIPS
trips_clean = trips_clean.drop_duplicates()

trips_clean["picked_ts"] = pd.to_datetime(
    trips_clean["picked_ts"],
    errors="coerce"
)

trips_clean["delivered_ts"] = pd.to_datetime(
    trips_clean["delivered_ts"],
    errors="coerce"
)

# Negative distance → positive distance
trips_clean["distance_km"] = trips_clean["distance_km"].abs()

print("Cleaning completed successfully!")

C:\Users\vaishuu\AppData\Local\Temp\ipykernel_22436\1054917877.py:18: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  stores_clean["opened_date"] = pd.to_datetime(
C:\Users\vaishuu\AppData\Local\Temp\ipykernel_22436\1054917877.py:90: UserWarning: Parsing dates in %d-%m-%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  trips_clean["picked_ts"] = pd.to_datetime(
C:\Users\vaishuu\AppData\Local\Temp\ipykernel_22436\1054917877.py:95: UserWarning: Parsing dates in %d-%m-%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  trips_clean["delivered_ts"] = pd.to_datetime(


Cleaning completed successfully!


In [27]:
print("Stores:", stores_clean.shape)
print("Order Items:", order_items_clean.shape)
print("Orders:", orders_final.shape)
print("Trips:", trips_clean.shape)

print("\nRemaining duplicate stores:",
      stores_clean.duplicated().sum())

print("Remaining duplicate order items:",
      order_items_clean.duplicated().sum())

print("Remaining duplicate trips:",
      trips_clean.duplicated().sum())

print("Remaining missing unit costs:",
      order_items_clean["unit_cost"].isna().sum())

print("Remaining negative distances:",
      (trips_clean["distance_km"] < 0).sum())

Stores: (10, 9)
Order Items: (177038, 6)
Orders: (52133, 14)
Trips: (52133, 7)

Remaining duplicate stores: 0
Remaining duplicate order items: 21
Remaining duplicate trips: 0
Remaining missing unit costs: 0
Remaining negative distances: 0


In [28]:
duplicate_item_ids = (
    order_items_clean[
        order_items_clean["order_item_id"].duplicated(keep=False)
    ]
    .sort_values("order_item_id")
)

duplicate_item_ids

,order_item_id,order_id,sku,quantity,unit_price,unit_cost
3091,I0003092,O000902,SKU0053,1,162.83,108.35
177019,I0003092,O000902,SKU0053,1,162.83,108.35
177020,I0004465,O001299,SKU0013,2,178.61,138.16
4464,I0004465,O001299,SKU0013,2,178.61,138.16
15885,I0015886,O004677,SKU0049,2,81.91,49.48
177030,I0015886,O004677,SKU0049,2,81.91,49.48
18464,I0018465,O005428,SKU0044,1,54.35,36.27
177022,I0018465,O005428,SKU0044,1,54.35,36.27
177033,I0022596,O006641,SKU0003,3,86.81,67.81
22595,I0022596,O006641,SKU0003,3,86.81,67.81


In [29]:
print("Duplicated order_item IDs:",
      duplicate_item_ids["order_item_id"].nunique())

print("Rows involved:",
      len(duplicate_item_ids))

Duplicated order_item IDs: 21
Rows involved: 42


In [30]:
duplicate_item_ids.groupby("order_item_id").size().value_counts()

2    21
Name: count, dtype: int64

In [31]:
duplicate_item_ids[
    ["order_item_id", "order_id", "sku", "quantity", "unit_price", "unit_cost"]
].head(50)

,order_item_id,order_id,sku,quantity,unit_price,unit_cost
3091,I0003092,O000902,SKU0053,1,162.83,108.35
177019,I0003092,O000902,SKU0053,1,162.83,108.35
177020,I0004465,O001299,SKU0013,2,178.61,138.16
4464,I0004465,O001299,SKU0013,2,178.61,138.16
15885,I0015886,O004677,SKU0049,2,81.91,49.48
177030,I0015886,O004677,SKU0049,2,81.91,49.48
18464,I0018465,O005428,SKU0044,1,54.35,36.27
177022,I0018465,O005428,SKU0044,1,54.35,36.27
177033,I0022596,O006641,SKU0003,3,86.81,67.81
22595,I0022596,O006641,SKU0003,3,86.81,67.81


In [32]:
# Remove duplicate order_item_id records
order_items_clean = order_items_clean.drop_duplicates(
    subset="order_item_id",
    keep="first"
)

print("Order items after cleaning:", order_items_clean.shape)
print(
    "Duplicate order_item_ids remaining:",
    order_items_clean["order_item_id"].duplicated().sum()
)

Order items after cleaning: (177017, 6)
Duplicate order_item_ids remaining: 0


In [33]:
cleaning_log.loc[len(cleaning_log)] = {
    "Dataset": "order_items_messy",
    "Column": "order_item_id",
    "Issue": "21 order_item_ids appeared twice after exact duplicate removal",
    "Rows_Affected": 21,
    "Action": "Removed one duplicate record per duplicated order_item_id",
    "Reason": "Each duplicated ID represented the same order, SKU, quantity, price and cost",
    "Impact_on_Analysis": "Prevents duplicated item quantities and costs from inflating product and COGS analysis"
}

In [34]:
cleaning_log

,Dataset,Column,Issue,Rows_Affected,Action,Reason,Impact_on_Analysis
0,dark_stores_messy,All columns,Duplicate store records,2,Remove exact duplicate records,S04 and S08 appear twice; store master should ...,Prevents duplicated store information from aff...
1,dark_stores_messy,staff_count,Missing values,2,Investigate before filling,Staff count is a store attribute; arbitrary im...,May affect orders-per-staff or productivity an...
2,dark_stores_messy,monthly_rent,Inconsistent text format,12,Convert to numeric,Values contain formats such as Rs. and INR,Required for store P&L calculation
3,dark_stores_messy,opened_date,Multiple date formats,12,Standardize to datetime,"Dates appear in formats such as YYYY-MM-DD, DD...",Ensures consistent date calculations
4,delivery_partners_messy,joined_date,Missing values,12,Investigate before filling,Missing joining dates cannot safely be inferred,May affect partner-tenure analysis
5,order_items_messy,All columns,Duplicate records,864,Investigate and remove confirmed duplicates,Duplicate order-item records can inflate quant...,Directly affects product sales and COGS analysis
6,order_items_messy,unit_cost,Missing values,2134,Reconcile with product master,Product master contains unit cost and can pote...,Affects product-level COGS calculation
7,orders_clean,delivered_ts,Missing values,1067,Investigate against order status,Missing delivery timestamps may be valid for c...,Important for delivery-time and SLA analysis
8,orders_clean,promo_code,Missing values,40067,Interpret as no promo only after validation,Missing promo code may represent orders withou...,Important for promotion impact analysis
9,trip_logs_messy,All columns,Duplicate records,312,Remove confirmed duplicate trip records,Trip table should have one trip record per trip,Prevents duplicate trips from distorting deliv...


In [35]:
print("===== FINAL CLEANING VALIDATION =====")

print("Stores:", stores_clean.shape)
print("Partners:", partners_clean.shape)
print("Order Items:", order_items_clean.shape)
print("Orders:", orders_final.shape)
print("Products:", products_clean.shape)
print("Promos:", promos_clean.shape)
print("Store Hours:", store_hours_clean.shape)
print("Trips:", trips_clean.shape)

print("\n===== DATA QUALITY =====")

print("Duplicate stores:",
      stores_clean.duplicated().sum())

print("Duplicate order items:",
      order_items_clean.duplicated().sum())

print("Duplicate order_item_ids:",
      order_items_clean["order_item_id"].duplicated().sum())

print("Duplicate trips:",
      trips_clean.duplicated().sum())

print("Missing order-item unit costs:",
      order_items_clean["unit_cost"].isna().sum())

print("Negative distances:",
      (trips_clean["distance_km"] < 0).sum())

===== FINAL CLEANING VALIDATION =====
Stores: (10, 9)
Partners: (124, 7)
Order Items: (177017, 6)
Orders: (52133, 14)
Products: (54, 7)
Promos: (6, 7)
Store Hours: (67, 4)
Trips: (52133, 7)

===== DATA QUALITY =====
Duplicate stores: 0
Duplicate order items: 0
Duplicate order_item_ids: 0
Duplicate trips: 0
Missing order-item unit costs: 0
Negative distances: 0


In [36]:
# ==========================================
# PHASE 2A - ORDER CONTRIBUTION
# ==========================================
# Calculate contribution per order
def calculate_contribution(row):
    if row["order_status"] == "delivered":
        return (
            row["net_amount"]
            - row["cogs_amount"]
            - row["delivery_cost"]
        )
    
    elif row["order_status"] == "returned":
        return -(
            row["cogs_amount"]
            + row["delivery_cost"]
        )
    
    elif row["order_status"] == "cancelled":
        return 0
    
    return 0


orders_final["contribution"] = orders_final.apply(
    calculate_contribution,
    axis=1
)

orders_final[
    [
        "order_id",
        "store_id",
        "order_status",
        "net_amount",
        "cogs_amount",
        "delivery_cost",
        "contribution"
    ]
].head()

,order_id,store_id,order_status,net_amount,cogs_amount,delivery_cost,contribution
0,O000001,S01,delivered,817.73,691.43,31.21,95.09
1,O000002,S01,delivered,313.15,203.48,21.06,88.61
2,O000003,S01,delivered,298.16,185.32,25.60,87.24
3,O000004,S01,delivered,734.37,493.21,21.68,219.48
4,O000005,S01,delivered,227.46,144.49,19.90,63.07


In [37]:
#Check the contribution calculation
print(
    orders_final.groupby("order_status")["contribution"]
    .agg(["count", "sum", "mean"])
)

              count         sum        mean
order_status                               
cancelled      1067        0.00    0.000000
delivered     48510  4163973.46   85.837424
returned       2556 -1138214.36 -445.310782


In [38]:
# Determine the months in the dataset

data_start = orders_final["order_ts"].min()
data_end = orders_final["order_ts"].max()

print("Data start:", data_start)
print("Data end:", data_end)

days_in_data = (
    data_end - data_start
).days + 1

months_in_data = days_in_data / 30

print("Days in data:", days_in_data)
print("Months in data:", months_in_data)

Data start: 2026-05-18 07:00:00
Data end: 2026-06-28 22:58:00
Days in data: 42
Months in data: 1.4


In [39]:
# Build the Store Performance table

store_summary = (
    orders_final
    .groupby("store_id")
    .agg(
        total_orders=("order_id", "count"),
        delivered_orders=(
            "order_status",
            lambda x: (x == "delivered").sum()
        ),
        returned_orders=(
            "order_status",
            lambda x: (x == "returned").sum()
        ),
        cancelled_orders=(
            "order_status",
            lambda x: (x == "cancelled").sum()
        ),
        net_revenue=("net_amount", "sum"),
        cogs=("cogs_amount", "sum"),
        delivery_cost=("delivery_cost", "sum"),
        contribution=("contribution", "sum")
    )
    .reset_index()
)

store_summary

,store_id,total_orders,delivered_orders,returned_orders,cancelled_orders,net_revenue,cogs,delivery_cost,contribution
0,S01,6091,5772,201,118,3215928.53,2465200.65,170790.95,459494.70
1,S02,5877,5584,178,115,3120772.81,2392695.97,164705.42,457583.42
2,S03,5564,5001,442,121,2773388.47,2234257.97,233600.32,79735.03
3,S04,5622,5333,166,123,2970023.40,2273753.03,157039.62,444250.81
4,S05,4688,4436,150,102,2470407.65,1889809.89,130964.33,359050.77
5,S06,6153,5811,210,132,3208854.45,2455151.40,172101.53,460390.89
6,S07,5687,4763,797,127,2838602.65,2396284.76,239530.15,-202668.46
7,S08,5309,5041,162,106,2791498.66,2130609.29,148857.98,423669.02
8,S09,1815,1712,72,31,964291.54,738397.88,50728.75,134691.69
9,S10,5327,5057,178,92,2807717.84,2144427.53,149503.58,409561.23


In [40]:
# Add rent and P&L

store_summary = store_summary.merge(
    stores_clean[
        [
            "store_id",
            "store_name",
            "monthly_rent",
            "sqft",
            "staff_count"
        ]
    ],
    on="store_id",
    how="left"
)

In [41]:
store_summary["rent_for_period"] = (
    store_summary["monthly_rent"] * months_in_data
)

store_summary["store_pnl"] = (
    store_summary["contribution"]
    - store_summary["rent_for_period"]
)

In [42]:
# Add the metrics that will help us find the root cause

store_summary["contribution_per_order"] = (
    store_summary["contribution"]
    / store_summary["total_orders"]
)

store_summary["return_rate"] = (
    store_summary["returned_orders"]
    / store_summary["total_orders"]
)

store_summary["cancellation_rate"] = (
    store_summary["cancelled_orders"]
    / store_summary["total_orders"]
)

store_summary["delivery_cost_per_order"] = (
    store_summary["delivery_cost"]
    / store_summary["total_orders"]
)

store_summary["net_revenue_per_order"] = (
    store_summary["net_revenue"]
    / store_summary["total_orders"]
)

In [43]:
store_summary = store_summary.sort_values(
    "store_pnl"
)

store_summary[
    [
        "store_id",
        "store_name",
        "total_orders",
        "contribution",
        "contribution_per_order",
        "rent_for_period",
        "store_pnl",
        "return_rate",
        "delivery_cost_per_order"
    ]
]

,store_id,store_name,total_orders,contribution,contribution_per_order,rent_for_period,store_pnl,return_rate,delivery_cost_per_order
6,S07,Sector 44 Metro,5687,-202668.46,-35.637148,319200.0,-521868.46,0.140144,42.118894
2,S03,Sector 137 Riverside,5564,79735.03,14.330523,274400.0,-194664.97,0.079439,41.984242
8,S09,Sector 150 Sports City,1815,134691.69,74.210298,240800.0,-106108.31,0.039669,27.949725
4,S05,Sector 76 Central,4688,359050.77,76.589328,253400.0,105650.77,0.031997,27.936077
9,S10,Sector 51 Junction,5327,409561.23,76.884030,231000.0,178561.23,0.033415,28.065249
0,S01,Sector 18 Hub,6091,459494.70,75.438302,249200.0,210294.70,0.033000,28.039887
7,S08,Sector 104 Gardens,5309,423669.02,79.802038,204400.0,219269.02,0.030514,28.038798
1,S02,Sector 62 Express,5877,457583.42,77.860034,226800.0,230783.42,0.030288,28.025425
3,S04,Greater Noida Alpha,5622,444250.81,79.020066,212800.0,231450.81,0.029527,27.933052
5,S06,Noida Extension North,6153,460390.89,74.823808,221200.0,239190.89,0.034130,27.970345


## Delivery Root Cause Analysis

In [44]:
# Delivery performance by store

delivery_analysis = orders_final[
    orders_final["order_status"].isin(["delivered", "returned"])
].copy()

delivery_analysis["delivery_minutes"] = (
    delivery_analysis["delivered_ts"] - delivery_analysis["order_ts"]
).dt.total_seconds() / 60

delivery_summary = delivery_analysis.groupby("store_id").agg(
    orders=("order_id", "count"),
    avg_delivery_minutes=("delivery_minutes", "mean"),
    median_delivery_minutes=("delivery_minutes", "median"),
    avg_promised_minutes=("promised_minutes", "mean"),
    avg_delivery_cost=("delivery_cost", "mean"),
    return_rate=("order_status", lambda x: (x == "returned").mean())
).reset_index()

delivery_summary["late_rate"] = (
    delivery_analysis.assign(
        late=delivery_analysis["delivery_minutes"] >
             delivery_analysis["promised_minutes"]
    )
    .groupby("store_id")["late"]
    .mean()
    .values
)

delivery_summary.sort_values("return_rate", ascending=False)

,store_id,orders,avg_delivery_minutes,median_delivery_minutes,avg_promised_minutes,avg_delivery_cost,return_rate,late_rate
6,S07,5560,15.717086,17.0,15.0,42.121772,0.143345,0.618705
2,S03,5443,15.947455,17.0,15.0,41.992535,0.081205,0.623369
8,S09,1784,14.566143,14.0,15.0,27.952578,0.040359,0.402466
5,S06,6021,14.609367,15.0,15.0,27.976153,0.034878,0.410895
9,S10,5235,14.570583,15.0,15.0,28.066953,0.034002,0.412607
0,S01,5973,14.575925,15.0,15.0,28.050025,0.033651,0.403650
4,S05,4586,14.525949,14.0,15.0,27.936424,0.032708,0.403184
7,S08,5203,14.616567,15.0,15.0,28.038017,0.031136,0.405343
1,S02,5762,14.609511,15.0,15.0,28.016336,0.030892,0.407150
3,S04,5499,14.697945,15.0,15.0,27.938296,0.030187,0.420986


In [45]:
# delivery partners

partner_analysis = orders_final.merge(
    trips_clean[[
        "order_id",
        "partner_id",
        "distance_km",
        "picked_ts",
        "delivered_ts"
    ]],
    on="order_id",
    how="left"
)

partner_analysis["delivery_minutes"] = (
    partner_analysis["delivered_ts_y"] -
    partner_analysis["picked_ts"]
).dt.total_seconds() / 60

partner_summary = partner_analysis.groupby(
    ["store_id", "partner_id"]
).agg(
    orders=("order_id", "count"),
    avg_delivery_minutes=("delivery_minutes", "mean"),
    avg_distance_km=("distance_km", "mean"),
    return_rate=("order_status", lambda x: (x == "returned").mean()),
    avg_delivery_cost=("delivery_cost", "mean")
).reset_index()

partner_summary[
    partner_summary["store_id"].isin(["S07", "S03"])
].sort_values(
    ["store_id", "return_rate"],
    ascending=[True, False]
)

,store_id,partner_id,orders,avg_delivery_minutes,avg_distance_km,return_rate,avg_delivery_cost
24,S03,P0025,469,6.656455,3.082964,0.181237,42.189275
22,S03,P0023,446,6.833333,2.979148,0.177130,41.937758
23,S03,P0024,453,7.182022,2.939073,0.161148,42.110993
25,S03,P0026,480,7.555085,3.033437,0.139583,41.643688
33,S03,P0034,457,14.055310,3.002932,0.043764,41.820175
31,S03,P0032,474,14.474836,3.018017,0.042194,42.147004
30,S03,P0031,501,14.298354,2.864611,0.041916,41.777705
29,S03,P0030,442,14.881944,2.974864,0.040724,41.704910
27,S03,P0028,447,14.474771,3.081029,0.038031,42.345168
32,S03,P0033,472,14.106061,2.857373,0.036017,42.260085


In [46]:
# promotion analysis

promo_analysis = orders_final.groupby(
    ["store_id", "promo_code"]
).agg(
    orders=("order_id", "count"),
    gross_amount=("gross_amount", "sum"),
    discount_amount=("discount_amount", "sum"),
    net_amount=("net_amount", "sum"),
    contribution=("contribution", "sum")
).reset_index()

promo_analysis[
    promo_analysis["store_id"].isin(["S07", "S03"])
].sort_values(
    ["store_id", "discount_amount"],
    ascending=[True, False]
)

,store_id,promo_code,orders,gross_amount,discount_amount,net_amount,contribution
15,S03,FRESH50,625,367413.46,183706.83,183706.63,-131580.42
18,S03,SAVE20,338,198283.10,39656.68,158626.42,-13946.60
19,S03,WEEKEND15,283,155472.45,23320.83,132151.62,-7125.66
14,S03,FLAT75,246,142401.59,18408.63,123992.96,1578.10
16,S03,NEW100,133,82502.75,13213.39,69289.36,-819.94
17,S03,NO_PROMO,3939,2105621.48,0.00,2105621.48,231629.55
40,S07,FRESH50,639,375719.40,187859.66,187859.74,-157466.27
43,S07,SAVE20,357,204576.16,40915.22,163660.94,-32877.50
44,S07,WEEKEND15,285,157537.93,23630.67,133907.26,-19548.65
39,S07,FLAT75,241,145762.85,18058.17,127704.68,-18487.68


## Network Benchmark

In [48]:
[name for name in dir() if not name.startswith("_")]

['In',
 'Out',
 'calculate_contribution',
 'cleaning_log',
 'data_end',
 'data_start',
 'datasets',
 'days_in_data',
 'delivery_analysis',
 'delivery_summary',
 'df',
 'duplicate_item_ids',
 'exit',
 'get_ipython',
 'missing',
 'missing_cost',
 'missing_promo',
 'months_in_data',
 'name',
 'negative_distance',
 'open',
 'order_items',
 'order_items_clean',
 'orders',
 'orders_final',
 'partner_analysis',
 'partner_summary',
 'partners',
 'partners_clean',
 'pd',
 'products',
 'products_clean',
 'promo_analysis',
 'promos',
 'promos_clean',
 'quit',
 'store_hours',
 'store_hours_clean',
 'store_summary',
 'stores',
 'stores_clean',
 'trips',
 'trips_clean']

In [49]:
store_summary.columns

Index(['store_id', 'total_orders', 'delivered_orders', 'returned_orders',
       'cancelled_orders', 'net_revenue', 'cogs', 'delivery_cost',
       'contribution', 'store_name', 'monthly_rent', 'sqft', 'staff_count',
       'rent_for_period', 'store_pnl', 'contribution_per_order', 'return_rate',
       'cancellation_rate', 'delivery_cost_per_order',
       'net_revenue_per_order'],
      dtype='object')

In [50]:
store_summary

,store_id,total_orders,delivered_orders,returned_orders,cancelled_orders,net_revenue,cogs,delivery_cost,contribution,store_name,monthly_rent,sqft,staff_count,rent_for_period,store_pnl,contribution_per_order,return_rate,cancellation_rate,delivery_cost_per_order,net_revenue_per_order
6,S07,5687,4763,797,127,2838602.65,2396284.76,239530.15,-202668.46,Sector 44 Metro,228000.0,2400,16.0,319200.0,-521868.46,-35.637148,0.140144,0.022332,42.118894,499.138852
2,S03,5564,5001,442,121,2773388.47,2234257.97,233600.32,79735.03,Sector 137 Riverside,196000.0,2100,13.0,274400.0,-194664.97,14.330523,0.079439,0.021747,41.984242,498.452277
8,S09,1815,1712,72,31,964291.54,738397.88,50728.75,134691.69,Sector 150 Sports City,172000.0,2150,13.0,240800.0,-106108.31,74.210298,0.039669,0.017080,27.949725,531.290105
4,S05,4688,4436,150,102,2470407.65,1889809.89,130964.33,359050.77,Sector 76 Central,181000.0,1800,11.0,253400.0,105650.77,76.589328,0.031997,0.021758,27.936077,526.964089
9,S10,5327,5057,178,92,2807717.84,2144427.53,149503.58,409561.23,Sector 51 Junction,165000.0,1850,12.0,231000.0,178561.23,76.884030,0.033415,0.017271,28.065249,527.072994
0,S01,6091,5772,201,118,3215928.53,2465200.65,170790.95,459494.70,Sector 18 Hub,178000.0,2200,14.0,249200.0,210294.70,75.438302,0.033000,0.019373,28.039887,527.980386
7,S08,5309,5041,162,106,2791498.66,2130609.29,148857.98,423669.02,Sector 104 Gardens,146000.0,1700,NaN,204400.0,219269.02,79.802038,0.030514,0.019966,28.038798,525.804984
1,S02,5877,5584,178,115,3120772.81,2392695.97,164705.42,457583.42,Sector 62 Express,162000.0,1900,12.0,226800.0,230783.42,77.860034,0.030288,0.019568,28.025425,531.014601
3,S04,5622,5333,166,123,2970023.40,2273753.03,157039.62,444250.81,Greater Noida Alpha,152000.0,2000,12.0,212800.0,231450.81,79.020066,0.029527,0.021878,27.933052,528.285912
5,S06,6153,5811,210,132,3208854.45,2455151.40,172101.53,460390.89,Noida Extension North,158000.0,2300,15.0,221200.0,239190.89,74.823808,0.034130,0.021453,27.970345,521.510556


In [51]:
# Stores that currently generate positive P&L
healthy_stores = store_summary[
    store_summary["store_pnl"] > 0
].copy()

# Median performance of healthy stores
healthy_benchmark = healthy_stores[
    [
        "contribution_per_order",
        "return_rate",
        "delivery_cost_per_order",
        "net_revenue_per_order"
    ]
].median()

print("HEALTHY STORE BENCHMARK")
print("-----------------------")
print(f"Contribution/order : ₹{healthy_benchmark['contribution_per_order']:.2f}")
print(f"Return rate        : {healthy_benchmark['return_rate']:.2%}")
print(f"Delivery cost/order: ₹{healthy_benchmark['delivery_cost_per_order']:.2f}")
print(f"Net revenue/order  : ₹{healthy_benchmark['net_revenue_per_order']:.2f}")

HEALTHY STORE BENCHMARK
-----------------------
Contribution/order : ₹76.88
Return rate        : 3.20%
Delivery cost/order: ₹28.03
Net revenue/order  : ₹527.07


In [52]:
# Compare S03 and S07 against healthy-store benchmark

problem_stores = store_summary[
    store_summary["store_id"].isin(["S03", "S07"])
].copy()

problem_stores["contribution_gap_per_order"] = (
    healthy_benchmark["contribution_per_order"]
    - problem_stores["contribution_per_order"]
)

problem_stores["excess_return_rate"] = (
    problem_stores["return_rate"]
    - healthy_benchmark["return_rate"]
)

problem_stores["excess_delivery_cost_per_order"] = (
    problem_stores["delivery_cost_per_order"]
    - healthy_benchmark["delivery_cost_per_order"]
)

problem_stores[
    [
        "store_id",
        "store_name",
        "total_orders",
        "store_pnl",
        "contribution_per_order",
        "contribution_gap_per_order",
        "return_rate",
        "excess_return_rate",
        "delivery_cost_per_order",
        "excess_delivery_cost_per_order"
    ]
]

,store_id,store_name,total_orders,store_pnl,contribution_per_order,contribution_gap_per_order,return_rate,excess_return_rate,delivery_cost_per_order,excess_delivery_cost_per_order
6,S07,Sector 44 Metro,5687,-521868.46,-35.637148,112.521178,0.140144,0.108148,42.118894,14.093469
2,S03,Sector 137 Riverside,5564,-194664.97,14.330523,62.553507,0.079439,0.047443,41.984242,13.958817


In [53]:
problem_stores["benchmark_contribution_gap"] = (
    problem_stores["contribution_per_order"]
    * 0  # placeholder
)

problem_stores["benchmark_contribution"] = (
    problem_stores["total_orders"]
    * healthy_benchmark["contribution_per_order"]
)

problem_stores["current_contribution"] = (
    problem_stores["contribution"]
)

problem_stores["theoretical_contribution_gap"] = (
    problem_stores["benchmark_contribution"]
    - problem_stores["current_contribution"]
)

problem_stores[
    [
        "store_id",
        "current_contribution",
        "benchmark_contribution",
        "theoretical_contribution_gap"
    ]
]

,store_id,current_contribution,benchmark_contribution,theoretical_contribution_gap
6,S07,-202668.46,437239.480948,639907.940948
2,S03,79735.03,427782.745207,348047.715207


In [54]:
orders_return_analysis = orders_final.copy()

orders_return_analysis["delivery_minutes"] = (
    orders_return_analysis["delivered_ts"]
    - orders_return_analysis["order_ts"]
).dt.total_seconds() / 60

orders_return_analysis["is_late"] = (
    orders_return_analysis["delivery_minutes"]
    > orders_return_analysis["promised_minutes"]
)

orders_return_analysis["is_returned"] = (
    orders_return_analysis["order_status"] == "returned"
)

orders_return_analysis["is_delivered"] = (
    orders_return_analysis["order_status"] == "delivered"
)

In [55]:
late_return_analysis = (
    orders_return_analysis[
        orders_return_analysis["order_status"].isin(
            ["delivered", "returned"]
        )
    ]
    .groupby(["store_id", "is_late"])
    .agg(
        orders=("order_id", "count"),
        returned_orders=("is_returned", "sum"),
        avg_delivery_minutes=("delivery_minutes", "mean")
    )
    .reset_index()
)

late_return_analysis["return_rate"] = (
    late_return_analysis["returned_orders"]
    / late_return_analysis["orders"]
)

late_return_analysis[
    late_return_analysis["store_id"].isin(["S03", "S07"])
]

,store_id,is_late,orders,returned_orders,avg_delivery_minutes,return_rate
4,S03,False,2050,317,8.879512,0.154634
5,S03,True,3393,125,20.217801,0.036841
12,S07,False,2120,498,8.403774,0.234906
13,S07,True,3440,299,20.224128,0.086919


In [56]:
# investigate the returned orders
returned_vs_delivered = (
    orders_return_analysis[
        orders_return_analysis["order_status"].isin(
            ["returned", "delivered"]
        )
    ]
    .groupby(["store_id", "order_status"])
    .agg(
        orders=("order_id", "count"),
        avg_delivery_minutes=("delivery_minutes", "mean"),
        median_delivery_minutes=("delivery_minutes", "median"),
        avg_promised_minutes=("promised_minutes", "mean"),
        avg_net_revenue=("net_amount", "mean"),
        avg_discount=("discount_amount", "mean"),
        avg_delivery_cost=("delivery_cost", "mean")
    )
    .reset_index()
)

returned_vs_delivered[
    returned_vs_delivered["store_id"].isin(["S03", "S07"])
]

,store_id,order_status,orders,avg_delivery_minutes,median_delivery_minutes,avg_promised_minutes,avg_net_revenue,avg_discount,avg_delivery_cost
4,S03,delivered,5001,16.539092,18.0,15.0,498.527704,49.176841,41.985739
5,S03,returned,442,9.253394,5.0,15.0,492.635068,62.641380,42.069434
12,S07,delivered,4763,16.491707,18.0,15.0,498.395864,48.688092,42.077747
13,S07,returned,797,11.087829,6.0,15.0,503.883087,56.815910,42.384868


In [57]:
# Check promotions
promo_return_analysis = (
    orders_return_analysis[
        orders_return_analysis["store_id"].isin(["S03", "S07"])
    ]
    .groupby(["store_id", "promo_code"])
    .agg(
        orders=("order_id", "count"),
        returned_orders=("is_returned", "sum"),
        avg_delivery_minutes=("delivery_minutes", "mean"),
        avg_discount=("discount_amount", "mean"),
        avg_net_revenue=("net_amount", "mean")
    )
    .reset_index()
)

promo_return_analysis["return_rate"] = (
    promo_return_analysis["returned_orders"]
    / promo_return_analysis["orders"]
)

promo_return_analysis.sort_values(
    ["store_id", "return_rate"],
    ascending=[True, False]
)

,store_id,promo_code,orders,returned_orders,avg_delivery_minutes,avg_discount,avg_net_revenue,return_rate
5,S03,WEEKEND15,283,28,16.212230,82.405760,466.966855,0.098940
1,S03,FRESH50,625,60,16.037277,293.930928,293.930608,0.096000
4,S03,SAVE20,338,26,15.742424,117.327456,469.308935,0.076923
3,S03,NO_PROMO,3939,301,15.976907,0.000000,534.557370,0.076415
2,S03,NEW100,133,10,15.543307,99.348797,520.972632,0.075188
0,S03,FLAT75,246,17,15.426160,74.831829,504.036423,0.069106
6,S07,FLAT75,241,44,15.579832,74.930166,529.894938,0.182573
8,S07,NEW100,141,24,15.808824,100.000000,507.179220,0.170213
7,S07,FRESH50,639,99,15.371383,293.990078,293.990203,0.154930
11,S07,WEEKEND15,285,42,15.649819,82.914632,469.850035,0.147368


In [58]:
#Check whether the strange return pattern is concentrated in specific partners

partner_return_analysis = (
    orders_return_analysis[
        orders_return_analysis["store_id"].isin(["S03", "S07"])
    ]
    .groupby(["store_id", "delivery_partner_id"])
    .agg(
        orders=("order_id", "count"),
        returned_orders=("is_returned", "sum"),
        avg_delivery_minutes=("delivery_minutes", "mean"),
        late_rate=("is_late", "mean"),
        avg_delivery_cost=("delivery_cost", "mean")
    )
    .reset_index()
)

partner_return_analysis["return_rate"] = (
    partner_return_analysis["returned_orders"]
    / partner_return_analysis["orders"]
)

partner_return_analysis.sort_values(
    ["store_id", "return_rate"],
    ascending=[True, False]
)

,store_id,delivery_partner_id,orders,returned_orders,avg_delivery_minutes,late_rate,avg_delivery_cost,return_rate
2,S03,P0025,469,85,10.693654,0.347548,42.189275,0.181237
0,S03,P0023,446,79,10.840183,0.345291,41.937758,0.177130
1,S03,P0024,453,73,11.148315,0.381898,42.110993,0.161148
3,S03,P0026,480,67,11.487288,0.360417,41.643688,0.139583
11,S03,P0034,457,20,18.117257,0.726477,41.820175,0.043764
9,S03,P0032,474,20,18.463895,0.736287,42.147004,0.042194
8,S03,P0031,501,21,18.362140,0.734531,41.777705,0.041916
7,S03,P0030,442,18,18.884259,0.735294,41.704910,0.040724
5,S03,P0028,447,17,18.550459,0.756152,42.345168,0.038031
10,S03,P0033,472,17,18.062771,0.707627,42.260085,0.036017


In [59]:
high_return_partners = {
    "S03": ["P0023", "P0024", "P0025", "P0026"],
    "S07": ["P0072", "P0073", "P0074", "P0075", "P0076"]
}

orders_return_analysis["high_return_partner_cohort"] = False

for store, partners_list in high_return_partners.items():
    mask = (
        (orders_return_analysis["store_id"] == store) &
        (orders_return_analysis["delivery_partner_id"].isin(partners_list))
    )
    
    orders_return_analysis.loc[
        mask, "high_return_partner_cohort"
    ] = True

In [60]:
partner_cohort_summary = (
    orders_return_analysis[
        orders_return_analysis["store_id"].isin(["S03", "S07"])
    ]
    .groupby(["store_id", "high_return_partner_cohort"])
    .agg(
        orders=("order_id", "count"),
        returned_orders=("is_returned", "sum"),
        avg_delivery_minutes=("delivery_minutes", "mean"),
        late_rate=("is_late", "mean"),
        avg_delivery_cost=("delivery_cost", "mean")
    )
    .reset_index()
)

partner_cohort_summary["return_rate"] = (
    partner_cohort_summary["returned_orders"]
    / partner_cohort_summary["orders"]
)

partner_cohort_summary

,store_id,high_return_partner_cohort,orders,returned_orders,avg_delivery_minutes,late_rate,avg_delivery_cost,return_rate
0,S03,False,3716,138,18.392729,0.734661,41.992481,0.037137
1,S03,True,1848,304,11.047461,0.358766,41.967673,0.164502
2,S07,False,3493,283,18.514060,0.748354,42.181924,0.081019
3,S07,True,2194,514,11.267474,0.376481,42.018546,0.234275


In [61]:
return_concentration = (
    orders_return_analysis[
        orders_return_analysis["store_id"].isin(["S03", "S07"])
        & orders_return_analysis["is_returned"]
    ]
    .groupby(["store_id", "high_return_partner_cohort"])
    .size()
    .reset_index(name="returned_orders")
)

return_concentration["share_of_store_returns"] = (
    return_concentration
    .groupby("store_id")["returned_orders"]
    .transform(lambda x: x / x.sum())
)

return_concentration

,store_id,high_return_partner_cohort,returned_orders,share_of_store_returns
0,S03,False,138,0.312217
1,S03,True,304,0.687783
2,S07,False,283,0.355082
3,S07,True,514,0.644918


In [62]:
partner_details = partners_clean[
    partners_clean["partner_id"].isin(
        ["P0023", "P0024", "P0025", "P0026",
         "P0072", "P0073", "P0074", "P0075", "P0076"]
    )
].copy()

partner_details

,partner_id,partner_name,joined_date,vehicle_type,home_store_id,employment_type,rating
22,P0023,Partner 0023,2025-07-14,EV Scooter,S03,Gig,3.9
23,P0024,Partner 0024,NaT,EV Scooter,S03,Gig,4.5
24,P0025,Partner 0025,2025-05-22,Petrol Scooter,S03,Gig,4.8 stars
25,P0026,Partner 0026,NaT,Petrol Scooter,S03,Full-time,3.5
71,P0072,Partner 0072,2026-02-18,Bicycle,S07,Gig,3.9
72,P0073,Partner 0073,2026-03-21,Petrol Scooter,S07,Gig,4.9 stars
73,P0074,Partner 0074,2025-07-31,EV Scooter,S07,Gig,4.5
74,P0075,Partner 0075,2025-08-14,EV Scooter,S07,Gig,4.1
75,P0076,Partner 0076,NaT,Petrol Scooter,S07,Gig,3.5 stars


In [63]:
status_check = orders_return_analysis.merge(
    trips_clean[
        [
            "order_id",
            "picked_ts",
            "delivered_ts",
            "trip_status"
        ]
    ],
    on="order_id",
    how="left",
    suffixes=("_order", "_trip")
)

status_check["order_to_pick_minutes"] = (
    status_check["picked_ts"] - status_check["order_ts"]
).dt.total_seconds() / 60

status_check["pick_to_trip_delivery_minutes"] = (
    status_check["delivered_ts_trip"] - status_check["picked_ts"]
).dt.total_seconds() / 60

status_check["order_to_trip_delivery_minutes"] = (
    status_check["delivered_ts_trip"] - status_check["order_ts"]
).dt.total_seconds() / 60

status_check["order_vs_trip_delivery_diff"] = (
    status_check["delivered_ts_order"] -
    status_check["delivered_ts_trip"]
).dt.total_seconds() / 60

In [64]:
pd.crosstab(
    status_check["order_status"],
    status_check["trip_status"]
)

trip_status,cancelled,delivered,returned
order_status,,,
cancelled,1067,0,0
delivered,0,48510,0
returned,0,0,2556


In [65]:
status_check[
    status_check["store_id"].isin(["S03", "S07"])
].groupby(
    ["store_id", "high_return_partner_cohort", "order_status"]
).agg(
    orders=("order_id", "count"),
    avg_order_to_pick=("order_to_pick_minutes", "mean"),
    avg_pick_to_delivery=("pick_to_trip_delivery_minutes", "mean"),
    avg_order_to_trip_delivery=("order_to_trip_delivery_minutes", "mean"),
    avg_delivery_difference=("order_vs_trip_delivery_diff", "mean")
).reset_index()

,store_id,high_return_partner_cohort,order_status,orders,avg_order_to_pick,avg_pick_to_delivery,avg_order_to_trip_delivery,avg_delivery_difference
0,S03,False,cancelled,85,4.047059,NaN,NaN,NaN
1,S03,False,delivered,3493,4.025193,14.373891,18.399084,0.0
2,S03,False,returned,138,4.028986,14.202899,18.231884,0.0
3,S03,True,cancelled,36,3.972222,NaN,NaN,NaN
4,S03,True,delivered,1508,4.009947,8.220822,12.230769,0.0
5,S03,True,returned,304,3.861842,1.315789,5.177632,0.0
6,S07,False,cancelled,79,4.177215,NaN,NaN,NaN
7,S07,False,delivered,3131,4.008943,14.491856,18.500798,0.0
8,S07,False,returned,283,4.000000,14.660777,18.660777,0.0
9,S07,True,cancelled,48,3.979167,NaN,NaN,NaN


In [66]:
# Focus only on S03 and S07
return_check = status_check[
    status_check["store_id"].isin(["S03", "S07"])
].copy()

# Compare returned orders by partner cohort
return_check[
    return_check["order_status"] == "returned"
].groupby(
    ["store_id", "high_return_partner_cohort"]
).agg(
    orders=("order_id", "count"),
    avg_order_to_pick=("order_to_pick_minutes", "mean"),
    avg_pick_to_delivery=("pick_to_trip_delivery_minutes", "mean"),
    avg_order_to_trip_delivery=("order_to_trip_delivery_minutes", "mean"),
    avg_order_delivery=("delivery_minutes", "mean"),
    avg_promised=("promised_minutes", "mean")
).reset_index()

,store_id,high_return_partner_cohort,orders,avg_order_to_pick,avg_pick_to_delivery,avg_order_to_trip_delivery,avg_order_delivery,avg_promised
0,S03,False,138,4.028986,14.202899,18.231884,18.231884,15.0
1,S03,True,304,3.861842,1.315789,5.177632,5.177632,15.0
2,S07,False,283,4.000000,14.660777,18.660777,18.660777,15.0
3,S07,True,514,4.044747,2.873541,6.918288,6.918288,15.0


In [67]:
return_check[
    return_check["order_status"] == "returned"
][[
    "store_id",
    "order_id",
    "high_return_partner_cohort",
    "order_ts",
    "picked_ts",
    "delivered_ts_order",
    "delivered_ts_trip",
    "order_to_pick_minutes",
    "pick_to_trip_delivery_minutes",
    "order_to_trip_delivery_minutes",
    "order_vs_trip_delivery_diff",
    "promised_minutes"
]].head(30)

,store_id,order_id,high_return_partner_cohort,order_ts,picked_ts,delivered_ts_order,delivered_ts_trip,order_to_pick_minutes,pick_to_trip_delivery_minutes,order_to_trip_delivery_minutes,order_vs_trip_delivery_diff,promised_minutes
261,S03,O000262,True,2026-05-18 20:38:00,2026-05-18 20:44:00,2026-05-18 20:44:00,2026-05-18 20:44:00,6.0,0.0,6.0,0.0,15
267,S03,O000268,True,2026-05-18 12:25:00,2026-05-18 12:31:00,2026-05-18 12:31:00,2026-05-18 12:31:00,6.0,0.0,6.0,0.0,15
276,S03,O000277,True,2026-05-18 12:28:00,2026-05-18 12:30:00,2026-05-18 12:50:00,2026-05-18 12:50:00,2.0,20.0,22.0,0.0,15
298,S03,O000299,True,2026-05-18 22:32:00,2026-05-18 22:37:00,2026-05-18 22:37:00,2026-05-18 22:37:00,5.0,0.0,5.0,0.0,15
300,S03,O000301,False,2026-05-18 20:41:00,2026-05-18 20:47:00,2026-05-18 20:57:00,2026-05-18 20:57:00,6.0,10.0,16.0,0.0,15
303,S03,O000304,True,2026-05-18 12:38:00,2026-05-18 12:41:00,2026-05-18 12:41:00,2026-05-18 12:41:00,3.0,0.0,3.0,0.0,15
307,S03,O000308,True,2026-05-18 18:05:00,2026-05-18 18:08:00,2026-05-18 18:08:00,2026-05-18 18:08:00,3.0,0.0,3.0,0.0,15
317,S03,O000318,True,2026-05-18 08:42:00,2026-05-18 08:47:00,2026-05-18 08:47:00,2026-05-18 08:47:00,5.0,0.0,5.0,0.0,15
338,S03,O000339,True,2026-05-18 19:11:00,2026-05-18 19:14:00,2026-05-18 19:14:00,2026-05-18 19:14:00,3.0,0.0,3.0,0.0,15
743,S07,O000744,True,2026-05-18 14:15:00,2026-05-18 14:19:00,2026-05-18 14:33:00,2026-05-18 14:33:00,4.0,14.0,18.0,0.0,15


In [68]:
partner_return_timing = return_check[
    return_check["order_status"] == "returned"
].groupby(
    ["store_id", "delivery_partner_id", "high_return_partner_cohort"]
).agg(
    returned_orders=("order_id", "count"),
    avg_pick_to_delivery=("pick_to_trip_delivery_minutes", "mean"),
    median_pick_to_delivery=("pick_to_trip_delivery_minutes", "median"),
    avg_order_to_delivery=("order_to_trip_delivery_minutes", "mean"),
    avg_promised=("promised_minutes", "mean")
).reset_index()

partner_return_timing.sort_values(
    ["store_id", "returned_orders"],
    ascending=[True, False]
)

,store_id,delivery_partner_id,high_return_partner_cohort,returned_orders,avg_pick_to_delivery,median_pick_to_delivery,avg_order_to_delivery,avg_promised
2,S03,P0025,True,85,1.552941,0.0,5.470588,15.0
0,S03,P0023,True,79,0.784810,0.0,4.582278,15.0
1,S03,P0024,True,73,0.739726,0.0,4.575342,15.0
3,S03,P0026,True,67,2.268657,0.0,6.164179,15.0
8,S03,P0031,False,21,13.857143,14.0,18.047619,15.0
9,S03,P0032,False,20,15.000000,15.0,18.600000,15.0
11,S03,P0034,False,20,14.350000,14.0,18.900000,15.0
7,S03,P0030,False,18,14.611111,14.0,18.333333,15.0
5,S03,P0028,False,17,14.941176,16.0,19.117647,15.0
10,S03,P0033,False,17,13.000000,14.0,16.823529,15.0


In [69]:
partner_status_timing = return_check.groupby(
    [
        "store_id",
        "delivery_partner_id",
        "high_return_partner_cohort",
        "order_status"
    ]
).agg(
    orders=("order_id", "count"),
    avg_order_to_pick=("order_to_pick_minutes", "mean"),
    avg_pick_to_delivery=("pick_to_trip_delivery_minutes", "mean"),
    median_pick_to_delivery=("pick_to_trip_delivery_minutes", "median"),
    avg_order_to_delivery=("order_to_trip_delivery_minutes", "mean"),
    median_order_to_delivery=("order_to_trip_delivery_minutes", "median"),
    avg_promised=("promised_minutes", "mean")
).reset_index()

partner_status_timing.sort_values(
    ["store_id", "delivery_partner_id", "order_status"]
)

,store_id,delivery_partner_id,high_return_partner_cohort,order_status,orders,avg_order_to_pick,avg_pick_to_delivery,median_pick_to_delivery,avg_order_to_delivery,median_order_to_delivery,avg_promised
0,S03,P0023,True,cancelled,8,3.500000,NaN,NaN,NaN,NaN,15.0
1,S03,P0023,True,delivered,359,4.052925,8.164345,9.0,12.217270,13.0,15.0
2,S03,P0023,True,returned,79,3.797468,0.784810,0.0,4.582278,4.0,15.0
3,S03,P0024,True,cancelled,8,3.750000,NaN,NaN,NaN,NaN,15.0
4,S03,P0024,True,delivered,372,3.991935,8.446237,10.5,12.438172,14.0,15.0
...,...,...,...,...,...,...,...,...,...,...,...
70,S07,P0083,False,delivered,392,3.992347,14.311224,15.0,18.303571,19.0,15.0
71,S07,P0083,False,returned,33,4.151515,15.030303,14.0,19.181818,19.0,15.0
72,S07,P0084,False,cancelled,11,4.090909,NaN,NaN,NaN,NaN,15.0
73,S07,P0084,False,delivered,376,3.954787,14.609043,15.0,18.563830,19.0,15.0


In [70]:
# Compare delivered vs returned for the high-return cohort only

cohort_status_comparison = return_check[
    return_check["high_return_partner_cohort"] == True
].groupby(
    ["store_id", "order_status"]
).agg(
    orders=("order_id", "count"),
    avg_order_to_pick=("order_to_pick_minutes", "mean"),
    avg_pick_to_delivery=("pick_to_trip_delivery_minutes", "mean"),
    median_pick_to_delivery=("pick_to_trip_delivery_minutes", "median"),
    avg_order_to_delivery=("order_to_trip_delivery_minutes", "mean"),
    median_order_to_delivery=("order_to_trip_delivery_minutes", "median"),
    avg_promised=("promised_minutes", "mean")
).reset_index()

cohort_status_comparison

,store_id,order_status,orders,avg_order_to_pick,avg_pick_to_delivery,median_pick_to_delivery,avg_order_to_delivery,median_order_to_delivery,avg_promised
0,S03,cancelled,36,3.972222,NaN,NaN,NaN,NaN,15.0
1,S03,delivered,1508,4.009947,8.220822,10.0,12.230769,13.5,15.0
2,S03,returned,304,3.861842,1.315789,0.0,5.177632,4.0,15.0
3,S07,cancelled,48,3.979167,NaN,NaN,NaN,NaN,15.0
4,S07,delivered,1632,4.030025,8.607230,10.0,12.637255,14.0,15.0
5,S07,returned,514,4.044747,2.873541,0.0,6.918288,5.0,15.0


In [71]:
# Calculate cohort vs non-cohort return rates
cohort_return_rates = return_check.groupby(
    ["store_id", "high_return_partner_cohort"]
).agg(
    orders=("order_id", "count"),
    returned_orders=("is_returned", "sum")
).reset_index()

cohort_return_rates["return_rate"] = (
    cohort_return_rates["returned_orders"] /
    cohort_return_rates["orders"]
)

cohort_return_rates

,store_id,high_return_partner_cohort,orders,returned_orders,return_rate
0,S03,False,3716,138,0.037137
1,S03,True,1848,304,0.164502
2,S07,False,3493,283,0.081019
3,S07,True,2194,514,0.234275


In [72]:
# Get non-cohort return rate for each store
non_cohort_rates = (
    cohort_return_rates[
        cohort_return_rates["high_return_partner_cohort"] == False
    ][["store_id", "return_rate"]]
    .rename(columns={"return_rate": "benchmark_return_rate"})
)

cohort_impact = cohort_return_rates.merge(
    non_cohort_rates,
    on="store_id",
    how="left"
)

cohort_impact["expected_returns_at_benchmark"] = (
    cohort_impact["orders"] *
    cohort_impact["benchmark_return_rate"]
)

cohort_impact["excess_returns"] = (
    cohort_impact["returned_orders"] -
    cohort_impact["expected_returns_at_benchmark"]
)

cohort_impact

,store_id,high_return_partner_cohort,orders,returned_orders,return_rate,benchmark_return_rate,expected_returns_at_benchmark,excess_returns
0,S03,False,3716,138,0.037137,0.037137,138.000000,0.000000
1,S03,True,1848,304,0.164502,0.037137,68.628633,235.371367
2,S07,False,3493,283,0.081019,0.081019,283.000000,0.000000
3,S07,True,2194,514,0.234275,0.081019,177.756084,336.243916


In [73]:
# Add financial fields to return_check
return_check["return_cost"] = (
    return_check["cogs_amount"] +
    return_check["delivery_cost"]
)

returned_cohort_cost = (
    return_check[
        (return_check["high_return_partner_cohort"] == True) &
        (return_check["order_status"] == "returned")
    ]
    .groupby("store_id")
    .agg(
        returned_orders=("order_id", "count"),
        avg_cogs=("cogs_amount", "mean"),
        avg_delivery_cost=("delivery_cost", "mean"),
        avg_return_cost=("return_cost", "mean"),
        total_return_cost=("return_cost", "sum")
    )
    .reset_index()
)

returned_cohort_cost

,store_id,returned_orders,avg_cogs,avg_delivery_cost,avg_return_cost,total_return_cost
0,S03,304,406.553191,41.921579,448.474770,136336.33
1,S07,514,425.694981,42.108054,467.803035,240450.76


In [74]:
cohort_impact_financial = cohort_impact[
    cohort_impact["high_return_partner_cohort"] == True
].merge(
    returned_cohort_cost,
    on="store_id",
    how="left"
)

cohort_impact_financial["estimated_excess_return_cost"] = (
    cohort_impact_financial["excess_returns"] *
    cohort_impact_financial["avg_return_cost"]
)

cohort_impact_financial

,store_id,high_return_partner_cohort,orders,returned_orders_x,return_rate,benchmark_return_rate,expected_returns_at_benchmark,excess_returns,returned_orders_y,avg_cogs,avg_delivery_cost,avg_return_cost,total_return_cost,estimated_excess_return_cost
0,S03,True,1848,304,0.164502,0.037137,68.628633,235.371367,304,406.553191,41.921579,448.474770,136336.33,105558.119645
1,S07,True,2194,514,0.234275,0.081019,177.756084,336.243916,514,425.694981,42.108054,467.803035,240450.76,157295.924601


In [75]:
cohort_impact_financial["monthly_opportunity"] = (
    cohort_impact_financial["estimated_excess_return_cost"] / 1.4
)

cohort_impact_financial[
    [
        "store_id",
        "excess_returns",
        "avg_return_cost",
        "estimated_excess_return_cost",
        "monthly_opportunity"
    ]
]

,store_id,excess_returns,avg_return_cost,estimated_excess_return_cost,monthly_opportunity
0,S03,235.371367,448.474770,105558.119645,75398.656890
1,S07,336.243916,467.803035,157295.924601,112354.231858


In [76]:
# Healthy-store benchmark
benchmark_cpo = 76.8840
benchmark_delivery_cost = 28.03
benchmark_return_rate = 0.0320

loss_decomposition = store_summary[
    store_summary["store_id"].isin(["S03", "S07"])
].copy()

# Delivery cost gap
loss_decomposition["delivery_cost_gap_per_order"] = (
    loss_decomposition["delivery_cost_per_order"]
    - benchmark_delivery_cost
)

loss_decomposition["delivery_cost_excess_period"] = (
    loss_decomposition["delivery_cost_gap_per_order"]
    * loss_decomposition["total_orders"]
)

# Return-rate gap
loss_decomposition["return_rate_gap"] = (
    loss_decomposition["return_rate"]
    - benchmark_return_rate
)

loss_decomposition[
    [
        "store_id",
        "total_orders",
        "return_rate",
        "delivery_cost_per_order",
        "delivery_cost_gap_per_order",
        "delivery_cost_excess_period",
        "return_rate_gap"
    ]
]

,store_id,total_orders,return_rate,delivery_cost_per_order,delivery_cost_gap_per_order,delivery_cost_excess_period,return_rate_gap
6,S07,5687,0.140144,42.118894,14.088894,80123.54,0.108144
2,S03,5564,0.079439,41.984242,13.954242,77641.40,0.047439


In [78]:
[name for name in dir() if not name.startswith("_")]

['In',
 'Out',
 'benchmark_cpo',
 'benchmark_delivery_cost',
 'benchmark_return_rate',
 'calculate_contribution',
 'cleaning_log',
 'cohort_impact',
 'cohort_impact_financial',
 'cohort_return_rates',
 'cohort_status_comparison',
 'data_end',
 'data_start',
 'datasets',
 'days_in_data',
 'delivery_analysis',
 'delivery_summary',
 'df',
 'duplicate_item_ids',
 'exit',
 'get_ipython',
 'healthy_benchmark',
 'healthy_stores',
 'high_return_partners',
 'late_return_analysis',
 'loss_decomposition',
 'mask',
 'missing',
 'missing_cost',
 'missing_promo',
 'months_in_data',
 'name',
 'negative_distance',
 'non_cohort_rates',
 'open',
 'order_items',
 'order_items_clean',
 'orders',
 'orders_final',
 'orders_return_analysis',
 'partner_analysis',
 'partner_cohort_summary',
 'partner_details',
 'partner_return_analysis',
 'partner_return_timing',
 'partner_status_timing',
 'partner_summary',
 'partners',
 'partners_clean',
 'partners_list',
 'pd',
 'problem_stores',
 'products',
 'products_cle

In [79]:
store_summary.columns.tolist()

['store_id',
 'total_orders',
 'delivered_orders',
 'returned_orders',
 'cancelled_orders',
 'net_revenue',
 'cogs',
 'delivery_cost',
 'contribution',
 'store_name',
 'monthly_rent',
 'sqft',
 'staff_count',
 'rent_for_period',
 'store_pnl',
 'contribution_per_order',
 'return_rate',
 'cancellation_rate',
 'delivery_cost_per_order',
 'net_revenue_per_order']

In [80]:
store_summary.head()

,store_id,total_orders,delivered_orders,returned_orders,cancelled_orders,net_revenue,cogs,delivery_cost,contribution,store_name,monthly_rent,sqft,staff_count,rent_for_period,store_pnl,contribution_per_order,return_rate,cancellation_rate,delivery_cost_per_order,net_revenue_per_order
6,S07,5687,4763,797,127,2838602.65,2396284.76,239530.15,-202668.46,Sector 44 Metro,228000.0,2400,16.0,319200.0,-521868.46,-35.637148,0.140144,0.022332,42.118894,499.138852
2,S03,5564,5001,442,121,2773388.47,2234257.97,233600.32,79735.03,Sector 137 Riverside,196000.0,2100,13.0,274400.0,-194664.97,14.330523,0.079439,0.021747,41.984242,498.452277
8,S09,1815,1712,72,31,964291.54,738397.88,50728.75,134691.69,Sector 150 Sports City,172000.0,2150,13.0,240800.0,-106108.31,74.210298,0.039669,0.017080,27.949725,531.290105
4,S05,4688,4436,150,102,2470407.65,1889809.89,130964.33,359050.77,Sector 76 Central,181000.0,1800,11.0,253400.0,105650.77,76.589328,0.031997,0.021758,27.936077,526.964089
9,S10,5327,5057,178,92,2807717.84,2144427.53,149503.58,409561.23,Sector 51 Junction,165000.0,1850,12.0,231000.0,178561.23,76.884030,0.033415,0.017271,28.065249,527.072994


In [81]:
store_performance = store_summary[
    [
        "store_id",
        "total_orders",
        "delivered_orders",
        "returned_orders",
        "cancelled_orders",
        "contribution",
        "rent_for_period",
        "store_pnl",
        "contribution_per_order",
        "return_rate",
        "cancellation_rate",
        "delivery_cost_per_order",
        "net_revenue_per_order"
    ]
].copy()

In [82]:
store_performance

,store_id,total_orders,delivered_orders,returned_orders,cancelled_orders,contribution,rent_for_period,store_pnl,contribution_per_order,return_rate,cancellation_rate,delivery_cost_per_order,net_revenue_per_order
6,S07,5687,4763,797,127,-202668.46,319200.0,-521868.46,-35.637148,0.140144,0.022332,42.118894,499.138852
2,S03,5564,5001,442,121,79735.03,274400.0,-194664.97,14.330523,0.079439,0.021747,41.984242,498.452277
8,S09,1815,1712,72,31,134691.69,240800.0,-106108.31,74.210298,0.039669,0.017080,27.949725,531.290105
4,S05,4688,4436,150,102,359050.77,253400.0,105650.77,76.589328,0.031997,0.021758,27.936077,526.964089
9,S10,5327,5057,178,92,409561.23,231000.0,178561.23,76.884030,0.033415,0.017271,28.065249,527.072994
0,S01,6091,5772,201,118,459494.70,249200.0,210294.70,75.438302,0.033000,0.019373,28.039887,527.980386
7,S08,5309,5041,162,106,423669.02,204400.0,219269.02,79.802038,0.030514,0.019966,28.038798,525.804984
1,S02,5877,5584,178,115,457583.42,226800.0,230783.42,77.860034,0.030288,0.019568,28.025425,531.014601
3,S04,5622,5333,166,123,444250.81,212800.0,231450.81,79.020066,0.029527,0.021878,27.933052,528.285912
5,S06,6153,5811,210,132,460390.89,221200.0,239190.89,74.823808,0.034130,0.021453,27.970345,521.510556


In [83]:
store_performance.shape

(10, 13)

In [84]:
store_performance.to_csv(
    "store_performance.csv",
    index=False
)

In [85]:
import os

os.path.exists("store_performance.csv")

True

In [86]:
store_performance[
    [
        "store_id",
        "total_orders",
        "store_pnl",
        "contribution_per_order",
        "return_rate",
        "delivery_cost_per_order"
    ]
]

,store_id,total_orders,store_pnl,contribution_per_order,return_rate,delivery_cost_per_order
6,S07,5687,-521868.46,-35.637148,0.140144,42.118894
2,S03,5564,-194664.97,14.330523,0.079439,41.984242
8,S09,1815,-106108.31,74.210298,0.039669,27.949725
4,S05,4688,105650.77,76.589328,0.031997,27.936077
9,S10,5327,178561.23,76.884030,0.033415,28.065249
0,S01,6091,210294.70,75.438302,0.033000,28.039887
7,S08,5309,219269.02,79.802038,0.030514,28.038798
1,S02,5877,230783.42,77.860034,0.030288,28.025425
3,S04,5622,231450.81,79.020066,0.029527,27.933052
5,S06,6153,239190.89,74.823808,0.034130,27.970345


In [87]:
import os

print(os.path.abspath("store_performance.csv"))

C:\Users\vaishuu\data_store_analysis\store_performance.csv


## Delivery Analysis

In [88]:
delivery_performance = delivery_analysis.copy()

delivery_performance

,order_id,store_id,customer_id,order_ts,delivered_ts,promised_minutes,order_status,promo_code,gross_amount,discount_amount,net_amount,cogs_amount,delivery_cost,delivery_partner_id,contribution,delivery_minutes
0,O000001,S01,C15543,2026-05-18 22:54:00,2026-05-18 23:08:00,15,delivered,NO_PROMO,817.73,0.0,817.73,691.43,31.21,P0001,95.09,14.0
1,O000002,S01,C13202,2026-05-18 18:39:00,2026-05-18 19:01:00,15,delivered,NO_PROMO,313.15,0.0,313.15,203.48,21.06,P0001,88.61,22.0
2,O000003,S01,C04044,2026-05-18 14:42:00,2026-05-18 14:56:00,15,delivered,NO_PROMO,298.16,0.0,298.16,185.32,25.60,P0011,87.24,14.0
3,O000004,S01,C04754,2026-05-18 18:10:00,2026-05-18 18:21:00,15,delivered,NO_PROMO,734.37,0.0,734.37,493.21,21.68,P0005,219.48,11.0
4,O000005,S01,C04934,2026-05-18 07:44:00,2026-05-18 07:52:00,15,delivered,NO_PROMO,227.46,0.0,227.46,144.49,19.90,P0004,63.07,8.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52128,O052129,S10,C12631,2026-06-28 07:41:00,2026-06-28 08:00:00,15,delivered,NO_PROMO,594.25,0.0,594.25,428.76,25.33,P0116,140.16,19.0
52129,O052130,S10,C17855,2026-06-28 21:39:00,2026-06-28 21:49:00,15,delivered,NO_PROMO,390.70,0.0,390.70,246.40,15.15,P0123,129.15,10.0
52130,O052131,S10,C11095,2026-06-28 18:20:00,2026-06-28 18:34:00,15,delivered,NO_PROMO,537.15,0.0,537.15,333.81,30.46,P0116,172.88,14.0
52131,O052132,S10,C02266,2026-06-28 20:13:00,2026-06-28 20:23:00,15,delivered,NO_PROMO,305.38,0.0,305.38,211.19,22.88,P0116,71.31,10.0


In [89]:
delivery_analysis.columns.tolist()

['order_id',
 'store_id',
 'customer_id',
 'order_ts',
 'delivered_ts',
 'promised_minutes',
 'order_status',
 'promo_code',
 'gross_amount',
 'discount_amount',
 'net_amount',
 'cogs_amount',
 'delivery_cost',
 'delivery_partner_id',
 'contribution',
 'delivery_minutes']

In [90]:
delivery_analysis.head()

,order_id,store_id,customer_id,order_ts,delivered_ts,promised_minutes,order_status,promo_code,gross_amount,discount_amount,net_amount,cogs_amount,delivery_cost,delivery_partner_id,contribution,delivery_minutes
0,O000001,S01,C15543,2026-05-18 22:54:00,2026-05-18 23:08:00,15,delivered,NO_PROMO,817.73,0.0,817.73,691.43,31.21,P0001,95.09,14.0
1,O000002,S01,C13202,2026-05-18 18:39:00,2026-05-18 19:01:00,15,delivered,NO_PROMO,313.15,0.0,313.15,203.48,21.06,P0001,88.61,22.0
2,O000003,S01,C04044,2026-05-18 14:42:00,2026-05-18 14:56:00,15,delivered,NO_PROMO,298.16,0.0,298.16,185.32,25.60,P0011,87.24,14.0
3,O000004,S01,C04754,2026-05-18 18:10:00,2026-05-18 18:21:00,15,delivered,NO_PROMO,734.37,0.0,734.37,493.21,21.68,P0005,219.48,11.0
4,O000005,S01,C04934,2026-05-18 07:44:00,2026-05-18 07:52:00,15,delivered,NO_PROMO,227.46,0.0,227.46,144.49,19.90,P0004,63.07,8.0


In [91]:
for name, value in globals().copy().items():
    if isinstance(value, pd.DataFrame):
        if "delivery_minutes" in value.columns and "delivery_partner_id" in value.columns:
            print(name, value.shape)

_ (5, 16)
___ (51066, 16)
delivery_analysis (51066, 16)
partner_analysis (52133, 20)
orders_return_analysis (52133, 20)
status_check (52133, 27)
return_check (11251, 28)
delivery_performance (51066, 16)
_88 (51066, 16)
_90 (5, 16)


In [92]:
print("Shape:", delivery_analysis.shape)
print("\nColumns:")
print(delivery_analysis.columns.tolist())

display(delivery_analysis.head())

Shape: (51066, 16)

Columns:
['order_id', 'store_id', 'customer_id', 'order_ts', 'delivered_ts', 'promised_minutes', 'order_status', 'promo_code', 'gross_amount', 'discount_amount', 'net_amount', 'cogs_amount', 'delivery_cost', 'delivery_partner_id', 'contribution', 'delivery_minutes']


,order_id,store_id,customer_id,order_ts,delivered_ts,promised_minutes,order_status,promo_code,gross_amount,discount_amount,net_amount,cogs_amount,delivery_cost,delivery_partner_id,contribution,delivery_minutes
0,O000001,S01,C15543,2026-05-18 22:54:00,2026-05-18 23:08:00,15,delivered,NO_PROMO,817.73,0.0,817.73,691.43,31.21,P0001,95.09,14.0
1,O000002,S01,C13202,2026-05-18 18:39:00,2026-05-18 19:01:00,15,delivered,NO_PROMO,313.15,0.0,313.15,203.48,21.06,P0001,88.61,22.0
2,O000003,S01,C04044,2026-05-18 14:42:00,2026-05-18 14:56:00,15,delivered,NO_PROMO,298.16,0.0,298.16,185.32,25.60,P0011,87.24,14.0
3,O000004,S01,C04754,2026-05-18 18:10:00,2026-05-18 18:21:00,15,delivered,NO_PROMO,734.37,0.0,734.37,493.21,21.68,P0005,219.48,11.0
4,O000005,S01,C04934,2026-05-18 07:44:00,2026-05-18 07:52:00,15,delivered,NO_PROMO,227.46,0.0,227.46,144.49,19.90,P0004,63.07,8.0


In [93]:
print("Order status:")
display(delivery_analysis["order_status"].value_counts(dropna=False))

print("\nStore-wise delivery summary:")
display(
    delivery_analysis.groupby("store_id").agg(
        total_orders=("order_id", "nunique"),
        avg_delivery_minutes=("delivery_minutes", "mean"),
        avg_promised_minutes=("promised_minutes", "mean"),
        avg_delivery_cost=("delivery_cost", "mean"),
        returned_orders=("order_status", lambda x: (x == "returned").sum()),
        cancelled_orders=("order_status", lambda x: (x == "cancelled").sum())
    ).round(2).sort_values("avg_delivery_minutes", ascending=False)
)

Order status:


order_status
delivered    48510
returned      2556
Name: count, dtype: int64


Store-wise delivery summary:


,total_orders,avg_delivery_minutes,avg_promised_minutes,avg_delivery_cost,returned_orders,cancelled_orders
store_id,,,,,,
S03,5443,15.95,15.0,41.99,442,0
S07,5560,15.72,15.0,42.12,797,0
S04,5499,14.70,15.0,27.94,166,0
S08,5203,14.62,15.0,28.04,162,0
S06,6021,14.61,15.0,27.98,210,0
S02,5762,14.61,15.0,28.02,178,0
S01,5973,14.58,15.0,28.05,201,0
S09,1784,14.57,15.0,27.95,72,0
S10,5235,14.57,15.0,28.07,178,0


In [94]:
import pandas as pd
import numpy as np

# Work on a copy
delivery_df = delivery_analysis.copy()

# Flag late deliveries against the promised time
delivery_df["is_late"] = (
    delivery_df["delivery_minutes"] > delivery_df["promised_minutes"]
)

# Flag returns
delivery_df["is_returned"] = (
    delivery_df["order_status"] == "returned"
)

# Overall KPIs
total_orders_in_analysis = delivery_df["order_id"].nunique()
avg_delivery_time = delivery_df["delivery_minutes"].mean()
avg_promised_time = delivery_df["promised_minutes"].mean()
late_delivery_rate = delivery_df["is_late"].mean() * 100
return_rate = delivery_df["is_returned"].mean() * 100
avg_delivery_cost = delivery_df["delivery_cost"].mean()

print("DELIVERY & FULFILMENT KPIs")
print("-" * 35)
print("Orders in delivery analysis:", f"{total_orders_in_analysis:,}")
print("Average delivery time:", round(avg_delivery_time, 2), "minutes")
print("Average promised time:", round(avg_promised_time, 2), "minutes")
print("Late delivery rate:", round(late_delivery_rate, 2), "%")
print("Return rate:", round(return_rate, 2), "%")
print("Average delivery cost per order: ₹", round(avg_delivery_cost, 2))

DELIVERY & FULFILMENT KPIs
-----------------------------------
Orders in delivery analysis: 51,066
Average delivery time: 14.87 minutes
Average promised time: 15.0 minutes
Late delivery rate: 45.46 %
Return rate: 5.01 %
Average delivery cost per order: ₹ 31.03


In [95]:
store_delivery_summary = (
    delivery_df.groupby("store_id")
    .agg(
        total_orders=("order_id", "nunique"),
        avg_delivery_minutes=("delivery_minutes", "mean"),
        avg_promised_minutes=("promised_minutes", "mean"),
        late_delivery_rate=("is_late", "mean"),
        return_rate=("is_returned", "mean"),
        avg_delivery_cost=("delivery_cost", "mean"),
        total_delivery_cost=("delivery_cost", "sum")
    )
    .reset_index()
)

store_delivery_summary["late_delivery_rate"] *= 100
store_delivery_summary["return_rate"] *= 100

store_delivery_summary = store_delivery_summary.round(2)

display(
    store_delivery_summary.sort_values(
        "late_delivery_rate", ascending=False
    )
)

,store_id,total_orders,avg_delivery_minutes,avg_promised_minutes,late_delivery_rate,return_rate,avg_delivery_cost,total_delivery_cost
2,S03,5443,15.95,15.0,62.34,8.12,41.99,228565.37
6,S07,5560,15.72,15.0,61.87,14.33,42.12,234197.05
3,S04,5499,14.70,15.0,42.10,3.02,27.94,153632.69
9,S10,5235,14.57,15.0,41.26,3.40,28.07,146930.50
5,S06,6021,14.61,15.0,41.09,3.49,27.98,168444.42
1,S02,5762,14.61,15.0,40.72,3.09,28.02,161430.13
7,S08,5203,14.62,15.0,40.53,3.11,28.04,145881.80
0,S01,5973,14.58,15.0,40.36,3.37,28.05,167542.80
4,S05,4586,14.53,15.0,40.32,3.27,27.94,128116.44
8,S09,1784,14.57,15.0,40.25,4.04,27.95,49867.40


In [96]:
# 1. Prepare order-level delivery data
delivery_export = delivery_df.copy()

delivery_export["is_late"] = delivery_export["is_late"].astype(int)
delivery_export["is_returned"] = delivery_export["is_returned"].astype(int)

# 2. Prepare store summary for Power BI
store_delivery_export = store_delivery_summary.copy()

# 3. Export both tables
delivery_export.to_csv("delivery_analysis_powerbi.csv", index=False)
store_delivery_export.to_csv("store_delivery_summary_powerbi.csv", index=False)

print("Files created successfully:")
print("- delivery_analysis_powerbi.csv")
print("- store_delivery_summary_powerbi.csv")

Files created successfully:
- delivery_analysis_powerbi.csv
- store_delivery_summary_powerbi.csv


In [97]:
print("Cleaned order-level data")
print("Rows:", len(delivery_analysis))
print("Unique orders:", delivery_analysis["order_id"].nunique())
print("\nStatus counts:")
print(delivery_analysis["order_status"].value_counts(dropna=False))

print("\nFull cleaned orders dataset")
print("Rows:", len(orders_final))
print("Unique orders:", orders_final["order_id"].nunique())
print("\nStatus counts:")
print(orders_final["order_status"].value_counts(dropna=False))

Cleaned order-level data
Rows: 51066
Unique orders: 51066

Status counts:
order_status
delivered    48510
returned      2556
Name: count, dtype: int64

Full cleaned orders dataset
Rows: 52133
Unique orders: 52133

Status counts:
order_status
delivered    48510
returned      2556
cancelled     1067
Name: count, dtype: int64


In [98]:
# Expected population: delivered + returned orders
expected_delivery_orders = orders_final[
    orders_final["order_status"].isin(["delivered", "returned"])
].copy()

# Compare unique order IDs
expected_ids = set(expected_delivery_orders["order_id"])
actual_ids = set(delivery_analysis["order_id"])

print("Expected delivered/returned orders:", len(expected_ids))
print("Actual delivery-analysis orders:", len(actual_ids))
print("Orders missing from delivery analysis:", len(expected_ids - actual_ids))
print("Unexpected orders in delivery analysis:", len(actual_ids - expected_ids))
print("Duplicate IDs in delivery analysis:",
      delivery_analysis["order_id"].duplicated().sum())

Expected delivered/returned orders: 51066
Actual delivery-analysis orders: 51066
Orders missing from delivery analysis: 0
Unexpected orders in delivery analysis: 0
Duplicate IDs in delivery analysis: 0


In [99]:
print("Raw trip dataset:")
print("Rows:", len(trips))

print("\nCleaned trip dataset:")
print("Rows:", len(trips_clean))

print("\nDuplicate order IDs in raw trips:",
      trips["order_id"].duplicated().sum())

print("Duplicate order IDs in cleaned trips:",
      trips_clean["order_id"].duplicated().sum())

Raw trip dataset:
Rows: 52445

Cleaned trip dataset:
Rows: 52133

Duplicate order IDs in raw trips: 312
Duplicate order IDs in cleaned trips: 0


In [100]:
print(
    "Cleaned trip rows equal unique order IDs:",
    len(trips_clean) == trips_clean["order_id"].nunique()
)

Cleaned trip rows equal unique order IDs: True


In [101]:
print("Partner analysis shape:", partner_analysis.shape)
print("Orders return analysis shape:", orders_return_analysis.shape)

print("\nPartner analysis columns:")
print(partner_analysis.columns.tolist())

print("\nOrders return analysis columns:")
print(orders_return_analysis.columns.tolist())

Partner analysis shape: (52133, 20)
Orders return analysis shape: (52133, 20)

Partner analysis columns:
['order_id', 'store_id', 'customer_id', 'order_ts', 'delivered_ts_x', 'promised_minutes', 'order_status', 'promo_code', 'gross_amount', 'discount_amount', 'net_amount', 'cogs_amount', 'delivery_cost', 'delivery_partner_id', 'contribution', 'partner_id', 'distance_km', 'picked_ts', 'delivered_ts_y', 'delivery_minutes']

Orders return analysis columns:
['order_id', 'store_id', 'customer_id', 'order_ts', 'delivered_ts', 'promised_minutes', 'order_status', 'promo_code', 'gross_amount', 'discount_amount', 'net_amount', 'cogs_amount', 'delivery_cost', 'delivery_partner_id', 'contribution', 'delivery_minutes', 'is_late', 'is_returned', 'is_delivered', 'high_return_partner_cohort']


In [102]:
display(partner_analysis.head())
display(orders_return_analysis.head())

,order_id,store_id,customer_id,order_ts,delivered_ts_x,promised_minutes,order_status,promo_code,gross_amount,discount_amount,net_amount,cogs_amount,delivery_cost,delivery_partner_id,contribution,partner_id,distance_km,picked_ts,delivered_ts_y,delivery_minutes
0,O000001,S01,C15543,2026-05-18 22:54:00,2026-05-18 23:08:00,15,delivered,NO_PROMO,817.73,0.0,817.73,691.43,31.21,P0001,95.09,P0001,5.03,2026-05-18 22:57:00,2026-05-18 23:08:00,11.0
1,O000002,S01,C13202,2026-05-18 18:39:00,2026-05-18 19:01:00,15,delivered,NO_PROMO,313.15,0.0,313.15,203.48,21.06,P0001,88.61,P0001,2.54,2026-05-18 18:45:00,2026-05-18 19:01:00,16.0
2,O000003,S01,C04044,2026-05-18 14:42:00,2026-05-18 14:56:00,15,delivered,NO_PROMO,298.16,0.0,298.16,185.32,25.60,P0011,87.24,P0011,1.12,2026-05-18 14:47:00,2026-05-18 14:56:00,9.0
3,O000004,S01,C04754,2026-05-18 18:10:00,2026-05-18 18:21:00,15,delivered,NO_PROMO,734.37,0.0,734.37,493.21,21.68,P0005,219.48,P0005,3.56,2026-05-18 18:14:00,2026-05-18 18:21:00,7.0
4,O000005,S01,C04934,2026-05-18 07:44:00,2026-05-18 07:52:00,15,delivered,NO_PROMO,227.46,0.0,227.46,144.49,19.90,P0004,63.07,P0004,3.68,2026-05-18 07:47:00,2026-05-18 07:52:00,5.0


,order_id,store_id,customer_id,order_ts,delivered_ts,promised_minutes,order_status,promo_code,gross_amount,discount_amount,net_amount,cogs_amount,delivery_cost,delivery_partner_id,contribution,delivery_minutes,is_late,is_returned,is_delivered,high_return_partner_cohort
0,O000001,S01,C15543,2026-05-18 22:54:00,2026-05-18 23:08:00,15,delivered,NO_PROMO,817.73,0.0,817.73,691.43,31.21,P0001,95.09,14.0,False,False,True,False
1,O000002,S01,C13202,2026-05-18 18:39:00,2026-05-18 19:01:00,15,delivered,NO_PROMO,313.15,0.0,313.15,203.48,21.06,P0001,88.61,22.0,True,False,True,False
2,O000003,S01,C04044,2026-05-18 14:42:00,2026-05-18 14:56:00,15,delivered,NO_PROMO,298.16,0.0,298.16,185.32,25.60,P0011,87.24,14.0,False,False,True,False
3,O000004,S01,C04754,2026-05-18 18:10:00,2026-05-18 18:21:00,15,delivered,NO_PROMO,734.37,0.0,734.37,493.21,21.68,P0005,219.48,11.0,False,False,True,False
4,O000005,S01,C04934,2026-05-18 07:44:00,2026-05-18 07:52:00,15,delivered,NO_PROMO,227.46,0.0,227.46,144.49,19.90,P0004,63.07,8.0,False,False,True,False


In [103]:
print("Cohort labels:")
print(orders_return_analysis["high_return_partner_cohort"].value_counts(dropna=False))

print("\nCohort labels by store and order status:")
display(
    orders_return_analysis
    .groupby(["store_id", "high_return_partner_cohort", "order_status"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

Cohort labels:
high_return_partner_cohort
False    48091
True      4042
Name: count, dtype: int64

Cohort labels by store and order status:


order_status,store_id,high_return_partner_cohort,cancelled,delivered,returned
0,S01,False,118,5772,201
1,S02,False,115,5584,178
2,S03,False,85,3493,138
3,S03,True,36,1508,304
4,S04,False,123,5333,166
5,S05,False,102,4436,150
6,S06,False,132,5811,210
7,S07,False,79,3131,283
8,S07,True,48,1632,514
9,S08,False,106,5041,162


In [104]:
print("Partner analysis unique orders:",
      partner_analysis["order_id"].nunique())

print("Partner analysis duplicate order IDs:",
      partner_analysis["order_id"].duplicated().sum())

print("Orders return analysis unique orders:",
      orders_return_analysis["order_id"].nunique())

print("Orders return analysis duplicate order IDs:",
      orders_return_analysis["order_id"].duplicated().sum())

Partner analysis unique orders: 52133
Partner analysis duplicate order IDs: 0
Orders return analysis unique orders: 52133
Orders return analysis duplicate order IDs: 0


In [106]:
print(trips_clean.columns.tolist())

['trip_id', 'order_id', 'partner_id', 'picked_ts', 'delivered_ts', 'distance_km', 'trip_status']


In [108]:
# Partner cohorts identified during our earlier analysis
s03_cohort_ids = ["P0023", "P0024", "P0025", "P0026"]
s07_cohort_ids = ["P0072", "P0073", "P0074", "P0075", "P0076"]

# Keep delivered and returned orders
cohort_check = orders_final[
    orders_final["order_status"].isin(["delivered", "returned"])
].copy()

# Join the cleaned trip records to orders
cohort_check = cohort_check.merge(
    trips_clean[["order_id", "partner_id"]],
    on="order_id",
    how="left",
    validate="one_to_one"
)

# Independently reconstruct the cohort flag
cohort_check["cohort_rebuilt"] = (
    ((cohort_check["store_id"] == "S03") &
     (cohort_check["partner_id"].isin(s03_cohort_ids)))
    |
    ((cohort_check["store_id"] == "S07") &
     (cohort_check["partner_id"].isin(s07_cohort_ids)))
)

# Check whether any orders lack a matched partner
print("Orders without a matched partner:",
      cohort_check["partner_id"].isna().sum())

# Count orders by store, cohort, and status
cohort_reconciliation = (
    cohort_check.groupby(
        ["store_id", "cohort_rebuilt", "order_status"]
    )
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

display(cohort_reconciliation)

Orders without a matched partner: 0


order_status,store_id,cohort_rebuilt,delivered,returned
0,S01,False,5772,201
1,S02,False,5584,178
2,S03,False,3493,138
3,S03,True,1508,304
4,S04,False,5333,166
5,S05,False,4436,150
6,S06,False,5811,210
7,S07,False,3131,283
8,S07,True,1632,514
9,S08,False,5041,162


In [109]:
# Calculate return rates and order counts by store and cohort
cohort_rates = (
    cohort_check.groupby(["store_id", "cohort_rebuilt"])
    .agg(
        orders=("order_id", "nunique"),
        returned_orders=("order_status", lambda s: (s == "returned").sum()),
        avg_return_cost=("contribution", lambda s: 0)  # placeholder; not used
    )
    .reset_index()
)

cohort_rates["return_rate_pct"] = (
    cohort_rates["returned_orders"] / cohort_rates["orders"] * 100
).round(2)

display(cohort_rates)

,store_id,cohort_rebuilt,orders,returned_orders,avg_return_cost,return_rate_pct
0,S01,False,5973,201,0,3.37
1,S02,False,5762,178,0,3.09
2,S03,False,3631,138,0,3.80
3,S03,True,1812,304,0,16.78
4,S04,False,5499,166,0,3.02
5,S05,False,4586,150,0,3.27
6,S06,False,6021,210,0,3.49
7,S07,False,3414,283,0,8.29
8,S07,True,2146,514,0,23.95
9,S08,False,5203,162,0,3.11


In [110]:
# 1. Calculate return cost only for returned orders
returned = cohort_check[
    cohort_check["order_status"] == "returned"
].copy()

returned["return_cost"] = (
    returned["cogs_amount"] + returned["delivery_cost"]
)

# 2. Summarize orders, returns, and return costs by store/cohort
cohort_summary = (
    cohort_check.groupby(["store_id", "cohort_rebuilt"])
    .agg(
        orders=("order_id", "nunique"),
        returned_orders=(
            "order_status",
            lambda s: (s == "returned").sum()
        )
    )
    .reset_index()
)

return_cost_summary = (
    returned.groupby(["store_id", "cohort_rebuilt"])
    .agg(
        total_return_cost=("return_cost", "sum"),
        avg_return_cost=("return_cost", "mean")
    )
    .reset_index()
)

cohort_summary = cohort_summary.merge(
    return_cost_summary,
    on=["store_id", "cohort_rebuilt"],
    how="left"
)

cohort_summary["return_rate"] = (
    cohort_summary["returned_orders"] / cohort_summary["orders"]
)

display(cohort_summary.round(2))

,store_id,cohort_rebuilt,orders,returned_orders,total_return_cost,avg_return_cost,return_rate
0,S01,False,5973,201,88031.54,437.97,0.03
1,S02,False,5762,178,78175.03,439.19,0.03
2,S03,False,3631,138,62898.75,455.79,0.04
3,S03,True,1812,304,136336.33,448.47,0.17
4,S04,False,5499,166,69110.82,416.33,0.03
5,S05,False,4586,150,66055.53,440.37,0.03
6,S06,False,6021,210,89502.58,426.20,0.03
7,S07,False,3414,283,134043.56,473.65,0.08
8,S07,True,2146,514,240450.76,467.80,0.24
9,S08,False,5203,162,64719.58,399.50,0.03


In [111]:
scenario_results = []

for store in ["S03", "S07"]:
    store_data = cohort_summary[
        cohort_summary["store_id"] == store
    ]

    other = store_data[store_data["cohort_rebuilt"] == False].iloc[0]
    cohort = store_data[store_data["cohort_rebuilt"] == True].iloc[0]

    expected_returns = cohort["orders"] * other["return_rate"]
    excess_returns = max(
        0,
        cohort["returned_orders"] - expected_returns
    )

    avg_cost = cohort["avg_return_cost"]

    estimated_42_day_opportunity = excess_returns * avg_cost
    estimated_monthly_opportunity = (
        estimated_42_day_opportunity / 42 * 30
    )

    scenario_results.append({
        "store_id": store,
        "cohort_orders": int(cohort["orders"]),
        "actual_returns": int(cohort["returned_orders"]),
        "baseline_return_rate_pct": round(
            other["return_rate"] * 100, 2
        ),
        "expected_returns_at_baseline": round(expected_returns, 1),
        "modeled_excess_returns": round(excess_returns, 1),
        "avg_cost_per_return": round(avg_cost, 2),
        "modeled_opportunity_42_days": round(
            estimated_42_day_opportunity, 2
        ),
        "modeled_opportunity_per_30_days": round(
            estimated_monthly_opportunity, 2
        )
    })

scenario_df = pd.DataFrame(scenario_results)
display(scenario_df)

,store_id,cohort_orders,actual_returns,baseline_return_rate_pct,expected_returns_at_baseline,modeled_excess_returns,avg_cost_per_return,modeled_opportunity_42_days,modeled_opportunity_per_30_days
0,S03,1812,304,3.80,68.9,235.1,448.47,105451.23,75322.31
1,S07,2146,514,8.29,177.9,336.1,467.80,157233.07,112309.33


In [112]:
print(
    scenario_df[
        [
            "store_id",
            "modeled_excess_returns",
            "avg_cost_per_return",
            "modeled_opportunity_42_days",
            "modeled_opportunity_per_30_days"
        ]
    ].to_string(index=False)
)

store_id  modeled_excess_returns  avg_cost_per_return  modeled_opportunity_42_days  modeled_opportunity_per_30_days
     S03                   235.1               448.47                    105451.23                         75322.31
     S07                   336.1               467.80                    157233.07                        112309.33


In [113]:

# Create a folder for the exported dashboard data
os.makedirs("lane2_exports", exist_ok=True)

# 1. Overall and store-level delivery analysis
delivery_analysis.to_csv(
    "lane2_exports/delivery_analysis.csv",
    index=False
)

# 2. Partner cohort return costs and return rates
cohort_summary.to_csv(
    "lane2_exports/cohort_summary.csv",
    index=False
)

# 3. Modeled monthly financial opportunity
scenario_df.to_csv(
    "lane2_exports/return_opportunity.csv",
    index=False
)

print("Lane 2 files exported successfully!")
print(os.listdir("lane2_exports"))

Lane 2 files exported successfully!
['cohort_summary.csv', 'delivery_analysis.csv', 'return_opportunity.csv']


# lane 3- product Analysis

In [114]:
import os

os.makedirs("lane3_exports", exist_ok=True)

# 1. Cleaned order items
order_items_clean.to_csv(
    "lane3_exports/order_items_clean.csv",
    index=False
)

# 2. Cleaned product master
products_clean.to_csv(
    "lane3_exports/products_clean.csv",
    index=False
)

# 3. Cleaned orders
orders_final.to_csv(
    "lane3_exports/orders_clean.csv",
    index=False
)

# 4. Cleaned promotion master
promos_clean.to_csv(
    "lane3_exports/promos_clean.csv",
    index=False
)

print("Lane 3 files exported successfully!")
print(os.listdir("lane3_exports"))

Lane 3 files exported successfully!
['orders_clean.csv', 'order_items_clean.csv', 'products_clean.csv', 'promos_clean.csv']


In [115]:
# Keep only the columns needed for analysis
items = order_items_clean.copy()
orders_data = orders_final.copy()
products_data = products_clean.copy()

# Check required columns before merging
print("Order item columns:", items.columns.tolist())
print("Order columns:", orders_data.columns.tolist())
print("Product columns:", products_data.columns.tolist())

Order item columns: ['order_item_id', 'order_id', 'sku', 'quantity', 'unit_price', 'unit_cost']
Order columns: ['order_id', 'store_id', 'customer_id', 'order_ts', 'delivered_ts', 'promised_minutes', 'order_status', 'promo_code', 'gross_amount', 'discount_amount', 'net_amount', 'cogs_amount', 'delivery_cost', 'delivery_partner_id', 'contribution']
Product columns: ['sku', 'product_name', 'category', 'is_perishable', 'shelf_life_days', 'mrp', 'unit_cost']


In [116]:
# Merge order items with order-level information
product_analysis = order_items_clean.merge(
    orders_final[
        ["order_id", "store_id", "order_status", "promo_code",
         "gross_amount", "discount_amount", "net_amount",
         "delivery_cost", "contribution"]
    ],
    on="order_id",
    how="left",
    validate="many_to_one"
)

# Add product details
product_analysis = product_analysis.merge(
    products_clean[
        ["sku", "product_name", "category", "is_perishable",
         "shelf_life_days"]
    ],
    on="sku",
    how="left",
    validate="many_to_one"
)

# Calculate item-level sales and cost
product_analysis["item_revenue"] = (
    product_analysis["quantity"] * product_analysis["unit_price"]
)

product_analysis["item_cogs"] = (
    product_analysis["quantity"] * product_analysis["unit_cost"]
)

# Summarise product performance
product_profitability = (
    product_analysis
    .groupby(["sku", "product_name", "category"], dropna=False)
    .agg(
        units_sold=("quantity", "sum"),
        orders=("order_id", "nunique"),
        item_revenue=("item_revenue", "sum"),
        item_cogs=("item_cogs", "sum")
    )
    .reset_index()
)

product_profitability["item_gross_profit"] = (
    product_profitability["item_revenue"]
    - product_profitability["item_cogs"]
)

product_profitability["item_margin_pct"] = np.where(
    product_profitability["item_revenue"] != 0,
    product_profitability["item_gross_profit"]
    / product_profitability["item_revenue"] * 100,
    np.nan
)

# Highest and lowest gross profit products
print("TOP 10 PRODUCTS BY ITEM GROSS PROFIT")
display(
    product_profitability
    .sort_values("item_gross_profit", ascending=False)
    .head(10)
    .round(2)
)

print("BOTTOM 10 PRODUCTS BY ITEM GROSS PROFIT")
display(
    product_profitability
    .sort_values("item_gross_profit")
    .head(10)
    .round(2)
)

TOP 10 PRODUCTS BY ITEM GROSS PROFIT


,sku,product_name,category,units_sold,orders,item_revenue,item_cogs,item_gross_profit,item_margin_pct
37,SKU0038,Digestive Biscuits 250g,Snacks & Packaged,5761,3989,999418.28,585259.99,414158.29,41.44
39,SKU0040,Namkeen Mix 400g,Snacks & Packaged,6079,4192,1034402.64,639510.80,394891.84,38.18
30,SKU0031,Green Tea 25 bags,beverages,5973,4052,936924.78,572153.67,364771.11,38.93
31,SKU0032,Cold Coffee 180ml,Beverages,5872,3997,966237.60,602936.96,363300.64,37.60
44,SKU0045,Garbage Bags 30pc,HOUSEHOLD,5999,4126,817663.70,487658.71,330004.99,40.36
52,SKU0053,Deodorant 150ml,PERSONAL CARE,5812,3985,946367.96,629730.20,316637.76,33.46
53,SKU0054,Shaving Gel 60g,Personal Care,6029,4072,805534.69,498236.56,307298.13,38.15
49,SKU0050,Toothpaste 150g,Personal Care,6018,4138,933451.98,634778.64,298673.34,32.00
47,SKU0048,Room Freshener 220ml,Household,5862,4012,742656.78,450963.66,291693.12,39.28
35,SKU0036,Potato Chips 52g,Snacks & Packaged,5724,3922,686937.24,397703.52,289233.72,42.10


BOTTOM 10 PRODUCTS BY ITEM GROSS PROFIT


,sku,product_name,category,units_sold,orders,item_revenue,item_cogs,item_gross_profit,item_margin_pct
34,SKU0035,Lemon Iced Tea 400ml,beverages,5974,4122,1055605.80,1150592.40,-94986.60,-9.00
18,SKU0019,Chicken Curry Cut 500g,meat & seafood,3727,2575,674624.27,735337.10,-60712.83,-9.00
22,SKU0023,Mutton Curry Cut 500g,meat & seafood,3785,2591,665781.50,725698.05,-59916.55,-9.00
15,SKU0016,Butter 100g,Dairy & Eggs,3770,2595,603426.20,657751.90,-54325.70,-9.00
29,SKU0030,Orange Juice 1L,Beverages,5737,3951,199360.75,217317.56,-17956.81,-9.01
27,SKU0028,Whole Wheat Bun 4pc,Bakery,3805,2619,167762.45,182868.30,-15105.85,-9.00
6,SKU0007,Coriander 100g,fruits & vegetables,3996,2694,183136.68,144695.16,38441.52,20.99
11,SKU0012,Toned Milk 500ml,Dairy & Eggs,3802,2588,146415.02,105657.58,40757.44,27.84
16,SKU0017,Cheese Slices 200g,DAIRY & EGGS,3719,2573,153334.37,111607.19,41727.18,27.21
0,SKU0001,Banana Robusta 1kg,FRUITS & VEGETABLES,3775,2576,217402.25,167534.50,49867.75,22.94


In [117]:

# Standardise category labels
product_analysis["category_clean"] = (
    product_analysis["category"]
    .astype("string")
    .str.strip()
    .str.title()
)

# Summarise category-level product economics
category_profitability = (
    product_analysis
    .groupby("category_clean", dropna=False)
    .agg(
        units_sold=("quantity", "sum"),
        orders=("order_id", "nunique"),
        item_revenue=("item_revenue", "sum"),
        item_cogs=("item_cogs", "sum"),
        products=("sku", "nunique")
    )
    .reset_index()
)

category_profitability["item_gross_profit"] = (
    category_profitability["item_revenue"]
    - category_profitability["item_cogs"]
)

category_profitability["item_margin_pct"] = np.where(
    category_profitability["item_revenue"] != 0,
    category_profitability["item_gross_profit"]
    / category_profitability["item_revenue"] * 100,
    np.nan
)

display(
    category_profitability
    .sort_values("item_gross_profit", ascending=False)
    .round(2)
)

,category_clean,units_sold,orders,item_revenue,item_cogs,products,item_gross_profit,item_margin_pct
7,Snacks & Packaged,40977,23080,4785010.23,2918303.85,7,1866706.38,39.01
6,Personal Care,35693,20661,3768770.61,2437901.88,6,1330868.73,35.31
4,Household,35227,20507,3098418.84,1911674.62,6,1186744.22,38.30
1,Beverages,41353,23232,4813438.31,3677221.44,7,1136216.87,23.61
3,Fruits & Vegetables,37642,21281,4002249.01,3062362.72,10,939886.29,23.48
2,Dairy & Eggs,29563,17635,3324207.22,2774362.42,8,549844.80,16.54
0,Bakery,18761,11879,1853711.65,1527120.63,5,326591.02,17.62
5,Meat & Seafood,18615,11785,2618415.19,2472055.27,5,146359.92,5.59


In [118]:

# Aggregate order-level performance by store and promotion
promo_analysis = (
    orders_final
    .groupby(["store_id", "promo_code"], dropna=False)
    .agg(
        orders=("order_id", "nunique"),
        gross_revenue=("gross_amount", "sum"),
        discount_amount=("discount_amount", "sum"),
        net_revenue=("net_amount", "sum"),
        contribution=("contribution", "sum"),
        returned_orders=("order_status",
                         lambda s: (s == "returned").sum()),
        cancelled_orders=("order_status",
                          lambda s: (s == "cancelled").sum())
    )
    .reset_index()
)

promo_analysis["contribution_per_order"] = (
    promo_analysis["contribution"] / promo_analysis["orders"]
)

promo_analysis["discount_per_order"] = (
    promo_analysis["discount_amount"] / promo_analysis["orders"]
)

promo_analysis["return_rate_pct"] = (
    promo_analysis["returned_orders"] / promo_analysis["orders"] * 100
)

# Focus on S03 and S07, our previously identified priority stores
display(
    promo_analysis[
        promo_analysis["store_id"].isin(["S03", "S07"])
    ]
    .sort_values(["store_id", "contribution_per_order"])
    .round(2)
)

,store_id,promo_code,orders,gross_revenue,discount_amount,net_revenue,contribution,returned_orders,cancelled_orders,contribution_per_order,discount_per_order,return_rate_pct
15,S03,FRESH50,625,367413.46,183706.83,183706.63,-131580.42,60,8,-210.53,293.93,9.60
18,S03,SAVE20,338,198283.10,39656.68,158626.42,-13946.60,26,8,-41.26,117.33,7.69
19,S03,WEEKEND15,283,155472.45,23320.83,132151.62,-7125.66,28,5,-25.18,82.41,9.89
16,S03,NEW100,133,82502.75,13213.39,69289.36,-819.94,10,6,-6.16,99.35,7.52
14,S03,FLAT75,246,142401.59,18408.63,123992.96,1578.10,17,9,6.42,74.83,6.91
17,S03,NO_PROMO,3939,2105621.48,0.00,2105621.48,231629.55,301,85,58.80,0.00,7.64
40,S07,FRESH50,639,375719.40,187859.66,187859.74,-157466.27,99,17,-246.43,293.99,15.49
43,S07,SAVE20,357,204576.16,40915.22,163660.94,-32877.50,50,5,-92.09,114.61,14.01
41,S07,NEW100,141,85612.27,14100.00,71512.27,-11352.73,24,5,-80.52,100.00,17.02
39,S07,FLAT75,241,145762.85,18058.17,127704.68,-18487.68,44,3,-76.71,74.93,18.26


In [119]:

# Summarise FRESH50 across all stores
fresh50 = orders_final[
    orders_final["promo_code"].eq("FRESH50")
].copy()

fresh50_summary = (
    fresh50
    .groupby("store_id")
    .agg(
        orders=("order_id", "nunique"),
        discounts=("discount_amount", "sum"),
        contribution=("contribution", "sum"),
        returned_orders=("order_status",
                         lambda s: (s == "returned").sum()),
        cancelled_orders=("order_status",
                          lambda s: (s == "cancelled").sum())
    )
    .reset_index()
)

fresh50_summary["contribution_per_order"] = (
    fresh50_summary["contribution"] / fresh50_summary["orders"]
)

fresh50_summary["return_rate_pct"] = (
    fresh50_summary["returned_orders"]
    / fresh50_summary["orders"] * 100
)

# The dataset covers approximately 42 days
days_in_data = 42

fresh50_summary["contribution_30_day_equivalent"] = (
    fresh50_summary["contribution"] * 30 / days_in_data
)

display(
    fresh50_summary
    .sort_values("contribution")
    .round(2)
)

# Business-wide totals
print("BUSINESS-WIDE FRESH50 SUMMARY")
print(f"Orders: {len(fresh50):,}")
print(f"Discounts: ₹{fresh50['discount_amount'].sum():,.2f}")
print(f"Contribution: ₹{fresh50['contribution'].sum():,.2f}")
print(
    "30-day equivalent contribution: "
    f"₹{fresh50['contribution'].sum() * 30 / days_in_data:,.2f}"
)

,store_id,orders,discounts,contribution,returned_orders,cancelled_orders,contribution_per_order,return_rate_pct,contribution_30_day_equivalent
6,S07,639,187859.66,-157466.27,99,17,-246.43,15.49,-112475.91
2,S03,625,183706.83,-131580.42,60,8,-210.53,9.60,-93986.01
1,S02,48,24519.00,-14925.19,2,0,-310.94,4.17,-10660.85
4,S05,36,17941.12,-12305.73,5,0,-341.83,13.89,-8789.81
0,S01,44,21163.01,-12049.81,1,0,-273.86,2.27,-8607.01
5,S06,39,18549.61,-10400.74,2,0,-266.69,5.13,-7429.10
7,S08,32,16164.38,-9540.92,2,0,-298.15,6.25,-6814.94
9,S10,36,17917.56,-8817.81,0,2,-244.94,0.00,-6298.44
3,S04,34,15874.79,-8255.58,0,1,-242.81,0.00,-5896.84
8,S09,6,2885.40,-1566.87,0,0,-261.14,0.00,-1119.19


BUSINESS-WIDE FRESH50 SUMMARY
Orders: 1,539
Discounts: ₹506,581.36
Contribution: ₹-366,909.34
30-day equivalent contribution: ₹-262,078.10


In [120]:
# Validate overall contribution and FRESH50 status mix
print("OVERALL CONTRIBUTION")
print(f"₹{orders_final['contribution'].sum():,.2f}")

print("\nFRESH50 STATUS MIX")
fresh50_status = (
    fresh50["order_status"]
    .value_counts(dropna=False)
    .rename_axis("order_status")
    .reset_index(name="orders")
)

fresh50_status["share_pct"] = (
    fresh50_status["orders"] / fresh50_status["orders"].sum() * 100
)

display(fresh50_status.round(2))

print("\nFRESH50 CONTRIBUTION BY ORDER STATUS")
display(
    fresh50.groupby("order_status")
    .agg(
        orders=("order_id", "nunique"),
        discounts=("discount_amount", "sum"),
        contribution=("contribution", "sum")
    )
    .reset_index()
    .round(2)
)

OVERALL CONTRIBUTION
₹3,025,759.10

FRESH50 STATUS MIX


,order_status,orders,share_pct
0,delivered,1340,87.07
1,returned,171,11.11
2,cancelled,28,1.82



FRESH50 CONTRIBUTION BY ORDER STATUS


,order_status,orders,discounts,contribution
0,cancelled,28,8774.94,0.00
1,delivered,1340,442170.72,-275875.72
2,returned,171,55635.70,-91033.62


In [122]:
from pathlib import Path

output_dir = Path("lane3_exports")
output_dir.mkdir(exist_ok=True)

product_profitability.to_csv(
    output_dir / "product_profitability.csv", index=False
)

category_profitability.to_csv(
    output_dir / "category_profitability.csv", index=False
)

promo_analysis.to_csv(
    output_dir / "promo_analysis.csv", index=False
)

fresh50_summary.to_csv(
    output_dir / "fresh50_summary.csv", index=False
)

print("Exported files:")
for file in sorted(output_dir.glob("*.csv")):
    print(file, "—", file.stat().st_size, "bytes")

Exported files:
lane3_exports\category_profitability.csv — 808 bytes
lane3_exports\fresh50_summary.csv — 921 bytes
lane3_exports\product_profitability.csv — 5996 bytes
lane3_exports\promo_analysis.csv — 7025 bytes
